# SAP–PLM Material Entity Resolution & Duplicate Detection

This notebook builds a domain-aware record-linkage pipeline between SAP material records and a PLM material master, then audits repeated technical profiles inside the PLM master.

### What the project does

- filters the modeling scope to fabric materials,
- normalizes identifiers and selected technical attributes,
- generates SAP→PLM candidates with text and structured retrieval channels,
- trains group-specific pairwise rerankers,
- applies an `AUTO / REVIEW / LOW_CONFIDENCE` decision layer,
- provides a separate fallback path for SAP materials without structured attributes,
- detects repeated exact technical profiles inside the PLM master and prioritizes cleanup candidates.

### Data note

The source files are intentionally **not included** in this repository because they may contain proprietary master data. The notebook expects them under `data/` (or via the `DATA_FOLDER` environment variable). Turkish source-system column names are kept where they are part of the external schema; code, comments, explanations, and derived variable names are otherwise written in English.

### AI assistance

ChatGPT was used as a coding and documentation assistant for refactoring, debugging, English translation, and repository preparation. The project logic, domain assumptions, validation choices, and final review remain the author's responsibility.


## 1. Setup and data loading

The notebook uses a project-relative `data/` directory by default. This removes machine-specific paths and makes the repository portable. Install dependencies from `requirements.txt` before running the notebook.


In [ ]:
import os
import re
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, top_k_accuracy_score

In [ ]:
data_dir = os.getenv("MATERIAL_DATA_DIR")

if not data_dir:
    raise EnvironmentError(
        "MATERIAL_DATA_DIR is not set. "
        "Set it to the folder containing the private source files."
    )

DATA_FOLDER = Path(data_dir)

print(f"Using private data directory: {DATA_FOLDER}")
print(f"Directory exists: {DATA_FOLDER.exists()}")

In [ ]:
INPUT_FILES = {
    "material_attributes": DATA_FOLDER / "MaterialAttributes_full.xlsx",
    "plm_work": DATA_FOLDER / "PLM Çalışması - Mara.xlsx",
    "plm_codes": DATA_FOLDER / "PLM_Codes.xlsx",
    "plm_multi_value": DATA_FOLDER / "PLM_MULTI_VAL_CHAR.xlsx",
}

for name, path in INPUT_FILES.items():
    print(f"{name}: {path.exists()} -> {path}")

In [ ]:
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


# Resolve project root whether the notebook is launched from the repository
# root or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# Private source data should live outside the GitHub repository.
#
# Preferred environment variable:
#   MATERIAL_DATA_DIR
#
# Backward-compatible fallback:
#   DATA_FOLDER
#
# Final fallback:
#   <project_root>/data
data_directory = (
    os.environ.get("MATERIAL_DATA_DIR")
    or os.environ.get("DATA_FOLDER")
)

if data_directory:
    DATA_DIR = Path(data_directory).expanduser()
else:
    DATA_DIR = PROJECT_ROOT / "data"


INPUT_FILES = {
    "material_attributes": DATA_DIR / "MaterialAttributes_full.xlsx",
    "sap_plm_mapping": DATA_DIR / "PLM Çalışması - Mara.xlsx",
    "plm_codes": DATA_DIR / "PLM_Codes.xlsx",
    "plm_multi_value": DATA_DIR / "PLM_MULTI_VAL_CHAR.xlsx",
}


# Validate source files before loading.
missing_files = [
    path
    for path in INPUT_FILES.values()
    if not path.exists()
]

if missing_files:
    missing_list = "\n".join(
        f"- {path}"
        for path in missing_files
    )

    raise FileNotFoundError(
        "Required input files were not found.\n\n"
        f"Current data directory:\n{DATA_DIR}\n\n"
        "Set the MATERIAL_DATA_DIR environment variable to the folder "
        "containing the private source files.\n\n"
        f"Missing files:\n{missing_list}"
    )


print(f"Using data directory: {DATA_DIR}")


# Load source datasets.
material_attributes = pd.read_excel(
    INPUT_FILES["material_attributes"],
    sheet_name="Data",
)

sap_plm_mapping = pd.read_excel(
    INPUT_FILES["sap_plm_mapping"],
    sheet_name="GenericArticle",
)

plm_codes = pd.read_excel(
    INPUT_FILES["plm_codes"],
    sheet_name="Data",
)

plm_multi_value_char = pd.read_excel(
    INPUT_FILES["plm_multi_value"],
    sheet_name="Data",
)


def normalize_plm_code(value):
    """Normalize PLM identifiers without discarding alphanumeric codes."""

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    if value.endswith(".0") and value[:-2].isdigit():
        value = value[:-2]

    return value


# Canonical SAP material identifiers.
sap_plm_mapping["material_id"] = (
    pd.to_numeric(
        sap_plm_mapping["Malzeme"],
        errors="coerce",
    )
    .astype("Int64")
    .astype("string")
)

material_attributes["material_id"] = (
    pd.to_numeric(
        material_attributes["Malzeme"],
        errors="coerce",
    )
    .astype("Int64")
    .astype("string")
)


# Canonical PLM identifiers.
sap_plm_mapping["plm_code"] = (
    sap_plm_mapping["PLM Kodu"]
    .apply(normalize_plm_code)
)

plm_codes["plm_code"] = (
    plm_codes["PLM Kodu"]
    .apply(normalize_plm_code)
)

plm_multi_value_char["plm_code"] = (
    plm_multi_value_char["plm_code"]
    .apply(normalize_plm_code)
)


# Keep only multi-value characteristic records linked to valid PLM codes.
valid_plm_codes = set(
    plm_codes["plm_code"].dropna()
)

plm_multi_value_valid = (
    plm_multi_value_char[
        plm_multi_value_char["plm_code"].isin(valid_plm_codes)
    ]
    .copy()
)


print("Material attributes:", material_attributes.shape)
print("SAP–PLM mapping:", sap_plm_mapping.shape)
print("PLM master:", plm_codes.shape)
print(
    "PLM multi-value characteristics:",
    plm_multi_value_valid.shape,
)

## 2. Initial coverage and modeling scope

The source contains records outside the target fabric domain. Strong description-based rules are used to keep footwear-upper and trim materials out of the fabric matching pipeline. These rules are deliberately conservative and should be reviewed when the source taxonomy changes.


In [ ]:
mapping_materials = set(sap_plm_mapping["material_id"].dropna())
attribute_materials = set(material_attributes["material_id"].dropna())
common_materials = mapping_materials & attribute_materials

sap_plm_mapping["has_plm"] = sap_plm_mapping["plm_code"].notna()

print("SAP materials in mapping:", len(mapping_materials))
print("SAP materials with attributes:", len(attribute_materials))
print("Attribute coverage:", f"{len(common_materials) / len(mapping_materials):.2%}")
print("PLM mapping coverage:", f"{sap_plm_mapping['has_plm'].mean():.2%}")

# Materials without structured attributes are the main group where text-based
# scope rules are needed.
missing_attribute_materials = sap_plm_mapping[
    ~sap_plm_mapping["material_id"].isin(attribute_materials)
].copy()

for frame in (sap_plm_mapping, missing_attribute_materials):
    frame["combined_description"] = (
        frame["Türkçe malzeme açıklaması"].fillna("").astype(str)
        + " "
        + frame["Türkçe malzeme Uzun açıklaması"].fillna("").astype(str)
    ).str.upper()

footwear_upper_keywords = ["SAYA", "AYAKKABI", "MOSTRA", "SALPA"]
trim_keywords = [
    "FERMUAR", "DÜĞME", "DUGME", "TOKA", "KORDON", "ŞERİT", "SERIT",
    "BİYE", "BIYE", "LASTİK", "LASTIK", "AKSESUAR"
]

footwear_upper_pattern = r"\b(?:" + "|".join(footwear_upper_keywords) + r")\b"
trim_pattern = r"\b(?:" + "|".join(trim_keywords) + r")\b"

sap_plm_mapping["is_footwear_upper"] = (
    sap_plm_mapping["combined_description"]
    .str.contains(footwear_upper_pattern, regex=True, na=False)
)

trim_candidates = missing_attribute_materials[
    missing_attribute_materials["combined_description"]
    .str.contains(trim_pattern, regex=True, na=False)
].copy()

excluded_materials = (
    set(sap_plm_mapping.loc[sap_plm_mapping["is_footwear_upper"], "material_id"])
    | set(trim_candidates["material_id"])
)

fabric_materials = sap_plm_mapping[
    ~sap_plm_mapping["material_id"].isin(excluded_materials)
].copy()

fabric_materials["has_attributes"] = fabric_materials["material_id"].isin(attribute_materials)
fabric_attributes = material_attributes[
    material_attributes["material_id"].isin(set(fabric_materials["material_id"]))
].copy()
fabric_attributes["field_name"] = (
    fabric_attributes["Dahili krkt.no."].astype("string").str.strip().str.upper()
)
materials_with_attributes = set(fabric_attributes["material_id"].dropna().astype("string"))

scope_summary = pd.Series({
    "all_sap_materials": sap_plm_mapping["material_id"].nunique(),
    "fabric_materials": fabric_materials["material_id"].nunique(),
    "excluded_footwear_upper": sap_plm_mapping.loc[
        sap_plm_mapping["is_footwear_upper"], "material_id"
    ].nunique(),
    "excluded_trim": trim_candidates["material_id"].nunique(),
    "fabric_with_plm": fabric_materials.loc[
        fabric_materials["plm_code"].notna(), "material_id"
    ].nunique(),
    "fabric_with_attributes": fabric_materials.loc[
        fabric_materials["has_attributes"], "material_id"
    ].nunique(),
})

display(scope_summary.to_frame("count"))


## 3. Material-group normalization and core feature preparation

The matching model is restricted to knitted, woven, and denim fabrics. Source-system group labels remain unchanged, while a stable internal code is added for blocking and model segmentation.


In [ ]:
MATERIAL_GROUP_CODE = {
    "ORME": "1020001",
    "DOKUMA": "1020002",
    "DENIM": "1030004",
}
GROUP_DISPLAY_NAME = {
    "1020001": "KNITTED",
    "1020002": "WOVEN",
    "1030004": "DENIM",
}

fabric_materials["material_group_name"] = (
    fabric_materials["Mal grubu"].astype("string").str.strip().str.upper()
)
fabric_materials["material_group_code"] = (
    fabric_materials["material_group_name"].map(MATERIAL_GROUP_CODE).astype("string")
)

plm_codes["material_group_name"] = (
    plm_codes["Mal grubu"].astype("string").str.strip().str.upper()
)
plm_codes["material_group_code"] = (
    plm_codes["material_group_name"].map(MATERIAL_GROUP_CODE).astype("string")
)

known_material_groups = (
    fabric_materials.loc[
        fabric_materials["plm_code"].notna(),
        ["material_id", "plm_code", "material_group_code", "material_group_name"],
    ]
    .drop_duplicates()
    .merge(
        plm_codes[["plm_code", "material_group_code", "material_group_name"]]
        .drop_duplicates("plm_code")
        .rename(columns={
            "material_group_code": "plm_material_group_code",
            "material_group_name": "plm_material_group_name",
        }),
        on="plm_code",
        how="left",
    )
)

# Use the PLM master group as the target blocking group when it is available.
known_material_groups["material_group_code"] = (
    known_material_groups["plm_material_group_code"]
    .fillna(known_material_groups["material_group_code"])
)
known_material_groups["material_group_name"] = (
    known_material_groups["plm_material_group_name"]
    .fillna(known_material_groups["material_group_name"])
)

model_material_groups = list(MATERIAL_GROUP_CODE.values())
known_material_groups_model = known_material_groups[
    known_material_groups["material_group_code"].isin(model_material_groups)
].copy()

print("Known mappings in modeling scope:", len(known_material_groups_model))
display(
    known_material_groups_model["material_group_code"]
    .map(GROUP_DISPLAY_NAME)
    .value_counts(dropna=False)
    .rename_axis("material_group")
    .to_frame("known_mappings")
)


### 3.1 Weight normalization

SAP weight values contain multiple scale conventions. Candidate matching therefore compares both the observed value and an SAP ÷ 1000 alternative, while rejecting explicit unit conflicts.


In [ ]:
# Inspect SAP weight values together with their units
weight_values = (
    fabric_attributes[
        fabric_attributes["Dahili krkt.no."].isin(
            ["WEIGHT", "WEIGHTUOMNAME"]
        )
    ]
    .pivot_table(
        index="material_id",
        columns="Dahili krkt.no.",
        values="Karakteristik değeri",
        aggfunc="first",
    )
    .reset_index()
)

# Rename the SAP weight field to a clear canonical name
weight_values = weight_values.rename(
    columns={
        "WEIGHT": "sap_weight_raw"
    }
)

# Validate expected columns
required_weight_columns = {
    "material_id",
    "sap_weight_raw",
    "WEIGHTUOMNAME",
}

missing_weight_columns = (
    required_weight_columns - set(weight_values.columns)
)

if missing_weight_columns:
    raise KeyError(
        "Expected SAP weight columns were not found: "
        f"{sorted(missing_weight_columns)}. "
        f"Available columns: {weight_values.columns.tolist()}"
    )

# Convert SAP weight values to numeric
weight_values["sap_weight_raw"] = pd.to_numeric(
    weight_values["sap_weight_raw"],
    errors="coerce",
)

print("SAP weight records:", len(weight_values))
print(
    "SAP records with numeric weight:",
    weight_values["sap_weight_raw"].notna().sum(),
)

display(weight_values.head(30))

display(
    weight_values["WEIGHTUOMNAME"]
    .value_counts(dropna=False)
    .head(30)
)


# Prepare SAP weight information
sap_weight_audit = weight_values[
    [
        "material_id",
        "sap_weight_raw",
        "WEIGHTUOMNAME",
    ]
].copy()

sap_weight_audit = sap_weight_audit.rename(
    columns={
        "WEIGHTUOMNAME": "sap_weight_unit_raw"
    }
)


# Normalize only clearly equivalent weight-unit labels
def normalize_weight_unit(value):
    """Normalize equivalent weight-unit labels while preserving unknown units."""

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    unit_mapping = {
        "GSM": "GSM",
        "G/M2": "GSM",
        "GR/M2": "GSM",
        "G/M²": "GSM",
        "GR/M²": "GSM",
        "VRDOZPERSQYARD": "VRDOZPERSQYARD",
    }

    return unit_mapping.get(value, value)


sap_weight_audit["sap_weight_unit"] = (
    sap_weight_audit["sap_weight_unit_raw"]
    .apply(normalize_weight_unit)
)


# Prepare PLM weight information
plm_weight_audit = plm_codes[
    [
        "plm_code",
        "Kumaş ağırlığı",
        "Kumaş ağırlığı birimi",
    ]
].copy()

plm_weight_audit["plm_weight"] = pd.to_numeric(
    plm_weight_audit["Kumaş ağırlığı"],
    errors="coerce",
)

plm_weight_audit["plm_weight_unit"] = (
    plm_weight_audit["Kumaş ağırlığı birimi"]
    .apply(normalize_weight_unit)
)


# Build known SAP-PLM weight pairs
known_weight_audit = (
    known_material_groups[
        [
            "material_id",
            "plm_code",
            "material_group_code",
            "material_group_name",
        ]
    ]
    .merge(
        sap_weight_audit,
        on="material_id",
        how="left",
    )
    .merge(
        plm_weight_audit,
        on="plm_code",
        how="left",
    )
)


# Basic audit summary
print(
    "Known SAP-PLM mappings:",
    len(known_weight_audit),
)

print(
    "Mappings with SAP weight:",
    known_weight_audit["sap_weight_raw"].notna().sum(),
)

print(
    "Mappings with PLM weight:",
    known_weight_audit["plm_weight"].notna().sum(),
)

print(
    "Mappings with both weights:",
    (
        known_weight_audit["sap_weight_raw"].notna()
        & known_weight_audit["plm_weight"].notna()
    ).sum(),
)

display(
    known_weight_audit[
        [
            "material_id",
            "plm_code",
            "sap_weight_raw",
            "sap_weight_unit",
            "plm_weight",
            "plm_weight_unit",
        ]
    ].head(30)
)

In [ ]:
# Compare SAP and PLM weights while accounting for SAP x1000 scaling
def calculate_weight_similarity(
    sap_weight,
    plm_weight,
    sap_unit=None,
    plm_unit=None
):
    if (
        pd.isna(sap_weight)
        or pd.isna(plm_weight)
        or sap_weight <= 0
        or plm_weight <= 0
    ):
        return pd.Series(
            [np.nan, np.nan, pd.NA]
        )

    # If both units exist and disagree, do not compare them directly
    if (
        pd.notna(sap_unit)
        and pd.notna(plm_unit)
        and sap_unit != plm_unit
    ):
        return pd.Series(
            [np.nan, np.nan, "unit_mismatch"]
        )

    candidates = {
        "same_scale": sap_weight,
        "sap_x1000": sap_weight / 1000
    }

    relative_errors = {
        scale: abs(value - plm_weight) / plm_weight
        for scale, value in candidates.items()
    }

    best_scale = min(
        relative_errors,
        key=relative_errors.get
    )

    best_error = relative_errors[best_scale]

    # 1 = perfect match, approaches 0 as the error increases
    similarity = 1 / (1 + best_error)

    return pd.Series(
        [
            similarity,
            best_error,
            best_scale
        ]
    )

# Calculate scale-aware weight similarity for known mappings
known_weight_audit[
    [
        "weight_similarity",
        "weight_relative_error",
        "weight_scale"
    ]
] = known_weight_audit.apply(
    lambda row: calculate_weight_similarity(
        row["sap_weight_raw"],
        row["plm_weight"],
        row["sap_weight_unit"],
        row["plm_weight_unit"]
    ),
    axis=1
)

# Evaluate weight agreement
comparable_weight_pairs = known_weight_audit[
    known_weight_audit["weight_similarity"].notna()
].copy()

print(
    "Comparable weight pairs:",
    len(comparable_weight_pairs)
)

print(
    "Exact / nearly exact (<= 1% error):",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.01).mean():.2%}"
)

print(
    "Within 5%:",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.05).mean():.2%}"
)

print(
    "Within 10%:",
    f"{comparable_weight_pairs['weight_relative_error'].le(0.10).mean():.2%}"
)

display(
    comparable_weight_pairs[
        "weight_scale"
    ].value_counts()
)

display(
    comparable_weight_pairs
    .groupby("material_group_name")
    .agg(
        comparable_pairs=("material_id", "size"),
        within_5_percent=(
            "weight_relative_error",
            lambda x: (x <= 0.05).mean()
        ),
        within_10_percent=(
            "weight_relative_error",
            lambda x: (x <= 0.10).mean()
        )
    )
)


### 3.2 Coloring, yarn-count, yarn-type, and weave-type features

These features are normalized conservatively: only clearly equivalent labels are collapsed, and missing values are kept as missing rather than treated as disagreement.


In [ ]:
def parse_yarn_count(value):
    if pd.isna(value):
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": pd.NA
        }

    value = str(value).strip().upper()

    # Normalize decimal separator
    value = value.replace(",", ".")

    # Remove extra spaces
    value = re.sub(r"\s+", " ", value)

    # Detect yarn count system
    unit = pd.NA

    if re.search(r"\bNE\b", value):
        unit = "NE"

    elif re.search(r"\bD\b", value):
        unit = "DENIER"

    # Extract count and optional ply
    match = re.search(
        r"(\d+(?:\.\d+)?)"
        r"(?:\s*/\s*(\d+(?:\.\d+)?))?",
        value
    )

    if not match:
        return {
            "count": np.nan,
            "ply": np.nan,
            "unit": unit
        }

    count = float(match.group(1))

    ply = (
        float(match.group(2))
        if match.group(2)
        else np.nan
    )

    return {
        "count": count,
        "ply": ply,
        "unit": unit
    }

# Compare parsed yarn counts conservatively
def calculate_yarn_count_match(sap_value, plm_value):

    sap = parse_yarn_count(sap_value)
    plm = parse_yarn_count(plm_value)

    if pd.isna(sap["count"]) or pd.isna(plm["count"]):
        return np.nan

    # Main yarn count must agree
    if sap["count"] != plm["count"]:
        return False

    # If both units are known, they must agree
    if (
        pd.notna(sap["unit"])
        and pd.notna(plm["unit"])
        and sap["unit"] != plm["unit"]
    ):
        return False

    # If both ply values are known, they must agree
    if (
        pd.notna(sap["ply"])
        and pd.notna(plm["ply"])
        and sap["ply"] != plm["ply"]
    ):
        return False

    # /1 and missing ply are treated as equivalent
    explicit_ply = (
        sap["ply"]
        if pd.notna(sap["ply"])
        else plm["ply"]
    )

    if pd.notna(explicit_ply) and explicit_ply != 1:
        # Do not assume that missing ply equals /2, /3, etc.
        if pd.isna(sap["ply"]) or pd.isna(plm["ply"]):
            return False

    return True

# Prepare SAP coloring values
sap_coloring = (
    fabric_attributes[
        fabric_attributes["field_name"] == "COLORINGID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .copy()
)

sap_coloring["coloring_value"] = (
    sap_coloring["Karakteristik değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sap_coloring_sets = (
    sap_coloring
    .groupby("material_id")["coloring_value"]
    .apply(set)
    .rename("sap_coloring_set")
    .reset_index()
)


In [ ]:
# Prepare PLM coloring values from the multi-value characteristic table
plm_coloring = (
    plm_multi_value_valid[
        plm_multi_value_valid["PLM Karakteristik Tanımı"]
        == "COLORINGID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .dropna(subset=["PLM Karakteristik Değeri"])
    .copy()
)

plm_coloring["coloring_value"] = (
    plm_coloring["PLM Karakteristik Değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_coloring_sets = (
    plm_coloring
    .groupby("plm_code")["coloring_value"]
    .apply(set)
    .rename("plm_coloring_set")
    .reset_index()
)

# Prepare SAP knitting yarn count
sap_yarn_count_knit = (
    fabric_attributes[
        fabric_attributes["field_name"] == "YARNCOUNT1KNITSID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_yarn_count"
        }
    )
)

sap_yarn_count_knit["sap_yarn_count"] = (
    sap_yarn_count_knit["sap_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Prepare PLM knitting yarn count
plm_yarn_count_knit = (
    plm_codes[
        [
            "plm_code",
            "1.İplik numarası örme"
        ]
    ]
    .copy()
    .rename(
        columns={
            "1.İplik numarası örme": "plm_yarn_count"
        }
    )
)

plm_yarn_count_knit["plm_yarn_count"] = (
    plm_yarn_count_knit["plm_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Prepare SAP knitting yarn type values
sap_yarn_type_knit = (
    fabric_attributes[
        fabric_attributes["field_name"] == "YARN1TYPEKNITSID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .copy()
)

sap_yarn_type_knit["yarn_type"] = (
    sap_yarn_type_knit["Karakteristik değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sap_yarn_type_sets = (
    sap_yarn_type_knit
    .groupby("material_id")["yarn_type"]
    .apply(set)
    .rename("sap_yarn_type_set")
    .reset_index()
)

# Prepare PLM knitting yarn type values
plm_yarn_type_knit = (
    plm_multi_value_valid[
        plm_multi_value_valid["PLM Karakteristik Tanımı"]
        == "YARN1TYPEKNITSID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .dropna(subset=["PLM Karakteristik Değeri"])
    .copy()
)

plm_yarn_type_knit["yarn_type"] = (
    plm_yarn_type_knit["PLM Karakteristik Değeri"]
    .astype("string")
    .str.strip()
    .str.upper()
)

plm_yarn_type_sets = (
    plm_yarn_type_knit
    .groupby("plm_code")["yarn_type"]
    .apply(set)
    .rename("plm_yarn_type_set")
    .reset_index()
)


In [ ]:
# Normalize only clearly equivalent yarn-type labels
yarn_type_mapping = {
    "VORTEX": "VORTEKS",
    "COMBEDCOMPACT": "COMPACTCOMBED"
}


def normalize_yarn_type(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    return yarn_type_mapping.get(
        value,
        value
    )

# Normalize SAP yarn-type sets
sap_yarn_type_sets_normalized = (
    sap_yarn_type_knit
    .assign(
        yarn_type_normalized=lambda df:
            df["yarn_type"].apply(normalize_yarn_type)
    )
    .groupby("material_id")["yarn_type_normalized"]
    .apply(set)
    .rename("sap_yarn_type_set")
    .reset_index()
)

# Normalize PLM yarn-type sets
plm_yarn_type_sets_normalized = (
    plm_yarn_type_knit
    .assign(
        yarn_type_normalized=lambda df:
            df["yarn_type"].apply(normalize_yarn_type)
    )
    .groupby("plm_code")["yarn_type_normalized"]
    .apply(set)
    .rename("plm_yarn_type_set")
    .reset_index()
)

# Prepare SAP weft yarn count values
sap_weft_yarn_count = (
    fabric_attributes[
        fabric_attributes["field_name"] == "WEFTYARNCOUNT1ID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_weft_yarn_count"
        }
    )
)

sap_weft_yarn_count["sap_weft_yarn_count"] = (
    sap_weft_yarn_count["sap_weft_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Prepare PLM weft yarn count values
plm_weft_yarn_count = (
    plm_codes[
        [
            "plm_code",
            "1.Atkı iplik numarası"
        ]
    ]
    .copy()
    .rename(
        columns={
            "1.Atkı iplik numarası":
            "plm_weft_yarn_count"
        }
    )
)

plm_weft_yarn_count["plm_weft_yarn_count"] = (
    plm_weft_yarn_count["plm_weft_yarn_count"]
    .astype("string")
    .str.strip()
    .str.upper()
)


In [ ]:
# Prepare SAP weave type
sap_weave_type = (
    fabric_attributes[
        fabric_attributes["field_name"] == "WEAVETYPE"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna()
    .drop_duplicates("material_id")
    .rename(
        columns={
            "Karakteristik değeri": "sap_weave_type"
        }
    )
)

sap_weave_type["sap_weave_type"] = (
    sap_weave_type["sap_weave_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Prepare PLM weave type
plm_weave_type = (
    plm_codes[
        ["plm_code", "Dokuma tipi"]
    ]
    .copy()
    .rename(
        columns={
            "Dokuma tipi": "plm_weave_type"
        }
    )
)

plm_weave_type["plm_weave_type"] = (
    plm_weave_type["plm_weave_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)


## 4. Candidate generation

Candidate generation uses three complementary channels:

1. character n-gram TF-IDF retrieval over descriptions and structured PLM text,
2. exact normalized fabric structure plus weight tolerance,
3. exact normalized fiber set plus weight tolerance.

The union is intentionally recall-oriented; the reranker handles precision later.


In [ ]:
# Normalize text for retrieval
def normalize_retrieval_text(value):
    if pd.isna(value):
        return ""

    value = str(value).upper().strip()

    # Normalize whitespace and punctuation
    value = re.sub(r"[^A-ZÇĞİÖŞÜ0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

# Build SAP retrieval text
sap_candidates_source = (
    fabric_materials[
        [
            "material_id",
            "Mal grubu",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması"
        ]
    ]
    .copy()
)

sap_candidates_source["material_group_code"] = (
    sap_candidates_source["Mal grubu"]
    .map({
        "ORME": "1020001",
        "DOKUMA": "1020002",
        "DENIM": "1030004"
    })
)

sap_candidates_source["retrieval_text"] = (
    sap_candidates_source[
        "Türkçe malzeme açıklaması"
    ].fillna("")
    + " "
    + sap_candidates_source[
        "Türkçe malzeme Uzun açıklaması"
    ].fillna("")
)

sap_candidates_source["retrieval_text"] = (
    sap_candidates_source["retrieval_text"]
    .apply(normalize_retrieval_text)
)

# Build PLM retrieval text
plm_candidates_source = (
    plm_codes[
        [
            "plm_code",
            "material_group_code",
            "Türkçe malzeme açıklaması",
            "Malzeme Türkçe Adı",
            "Malzeme ingilizce adı"
        ]
    ]
    .copy()
)

plm_candidates_source["retrieval_text"] = (
    plm_candidates_source[
        "Türkçe malzeme açıklaması"
    ].fillna("")
    + " "
    + plm_candidates_source[
        "Malzeme Türkçe Adı"
    ].fillna("")
    + " "
    + plm_candidates_source[
        "Malzeme ingilizce adı"
    ].fillna("")
)

plm_candidates_source["retrieval_text"] = (
    plm_candidates_source["retrieval_text"]
    .apply(normalize_retrieval_text)
)

# PLM fields useful for candidate retrieval
plm_retrieval_fields = [
    "Kumaş tipi",
    "Dokuma tipi",
    "Örme alt tipi",
    "1.İplik numarası örme",
    "1.Atkı iplik numarası",
    "1.Çözgü iplik numarası",
    "Kumaş ağırlığı"
]


def combine_retrieval_fields(row, fields):
    values = []

    for field in fields:
        value = row.get(field)

        if pd.notna(value):
            value = normalize_retrieval_text(value)

            if value:
                values.append(value)

    return " ".join(values)


plm_structured_text = plm_codes[
    ["plm_code"] + plm_retrieval_fields
].copy()

plm_structured_text["structured_text"] = (
    plm_structured_text.apply(
        lambda row: combine_retrieval_fields(
            row,
            plm_retrieval_fields
        ),
        axis=1
    )
)


In [ ]:
# Characteristics to include in retrieval representation
retrieval_multi_fields = [
    "FIBERCONTENTLISTID",
    "COLORINGID",
    "YARN1TYPEKNITSID"
]

plm_multi_retrieval = (
    plm_multi_value_valid[
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ].isin(retrieval_multi_fields)
    ]
    .copy()
)

plm_multi_retrieval["retrieval_value"] = (
    plm_multi_retrieval[
        "PLM Karakteristik Değeri"
    ]
    .apply(normalize_retrieval_text)
)

plm_multi_text = (
    plm_multi_retrieval
    .groupby("plm_code")["retrieval_value"]
    .apply(
        lambda values:
            " ".join(sorted(set(values)))
    )
    .rename("multi_value_text")
    .reset_index()
)

# Enrich PLM retrieval text with structured characteristics
plm_candidates_enriched = (
    plm_candidates_source
    .merge(
        plm_structured_text[
            ["plm_code", "structured_text"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        plm_multi_text,
        on="plm_code",
        how="left"
    )
)

plm_candidates_enriched["structured_text"] = (
    plm_candidates_enriched[
        "structured_text"
    ].fillna("")
)

plm_candidates_enriched["multi_value_text"] = (
    plm_candidates_enriched[
        "multi_value_text"
    ].fillna("")
)

plm_candidates_enriched["retrieval_text_enriched"] = (
    plm_candidates_enriched["retrieval_text"]
    + " "
    + plm_candidates_enriched["structured_text"]
    + " "
    + plm_candidates_enriched["multi_value_text"]
).str.strip()


In [ ]:
# Generate top-k PLM candidates using character n-gram TF-IDF
def generate_tfidf_candidates(
    sap_df,
    plm_df,
    material_group_code,
    top_k=50
):
    sap_group = (
        sap_df[
            sap_df["material_group_code"] == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    plm_group = (
        plm_df[
            plm_df["material_group_code"] == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Keep only rows with usable retrieval text
    sap_group = sap_group[
        sap_group["retrieval_text"].str.len() > 0
    ].reset_index(drop=True)

    plm_group = plm_group[
        plm_group["retrieval_text_enriched"].str.len() > 0
    ].reset_index(drop=True)

    # Character n-grams are robust to spelling and formatting differences
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        sublinear_tf=True,
        norm="l2"
    )

    plm_matrix = vectorizer.fit_transform(
        plm_group["retrieval_text_enriched"]
    )

    sap_matrix = vectorizer.transform(
        sap_group["retrieval_text"]
    )

    n_neighbors = min(
        top_k,
        len(plm_group)
    )

    nn_model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="cosine",
        algorithm="brute"
    )

    nn_model.fit(plm_matrix)

    distances, indices = nn_model.kneighbors(
        sap_matrix
    )

    candidate_records = []

    for sap_idx in range(len(sap_group)):
        for rank, (plm_idx, distance) in enumerate(
            zip(indices[sap_idx], distances[sap_idx]),
            start=1
        ):
            candidate_records.append({
                "material_id":
                    sap_group.loc[sap_idx, "material_id"],
                "material_group_code":
                    material_group_code,
                "candidate_plm_code":
                    plm_group.loc[plm_idx, "plm_code"],
                "candidate_rank":
                    rank,
                "retrieval_similarity":
                    1 - distance
            })

    return pd.DataFrame(candidate_records)

# Generate candidates for each material group
candidate_tables = []

for material_group_code in [
    "1020001",  # ORME
    "1020002",  # DOKUMA
    "1030004"   # DENIM
]:
    group_candidates = generate_tfidf_candidates(
        sap_candidates_source,
        plm_candidates_enriched,
        material_group_code=material_group_code,
        top_k=50
    )

    candidate_tables.append(
        group_candidates
    )

tfidf_candidates = pd.concat(
    candidate_tables,
    ignore_index=True
)

print(
    "Candidate pairs:",
    len(tfidf_candidates)
)

display(
    tfidf_candidates.head(10)
)


In [ ]:
# Prepare known SAP-PLM mappings for retrieval evaluation
known_retrieval_pairs = (
    known_material_groups_model[
        [
            "material_id",
            "plm_code",
            "material_group_code"
        ]
    ]
    .dropna(subset=["plm_code"])
    .copy()
)

retrieval_evaluation = (
    known_retrieval_pairs
    .merge(
        tfidf_candidates,
        left_on=[
            "material_id",
            "material_group_code",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "material_group_code",
            "candidate_plm_code"
        ],
        how="left"
    )
)

# Normalize fabric structure for candidate generation
def normalize_fabric_structure(value):
    if pd.isna(value):
        return pd.NA

    value = (
        str(value)
        .strip()
        .upper()
    )

    mapping = {
        "POLYVISCON": "POLYVISCOSE",
        "BEZAYAĞI": "PLAINWEAVE"
    }

    return mapping.get(value, value)

# Prepare SAP fabric structures
sap_structure_candidates = (
    fabric_attributes[
        fabric_attributes["field_name"]
        == "FABRICSTRUCTURETNAME"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .dropna(subset=["Karakteristik değeri"])
    .drop_duplicates("material_id")
    .copy()
)

sap_structure_candidates["structure"] = (
    sap_structure_candidates[
        "Karakteristik değeri"
    ]
    .apply(normalize_fabric_structure)
)

# Prepare PLM fabric structures
plm_structure_candidates = (
    plm_codes[
        [
            "plm_code",
            "material_group_code",
            "Kumaş tipi"
        ]
    ]
    .dropna(subset=["Kumaş tipi"])
    .copy()
)

plm_structure_candidates["structure"] = (
    plm_structure_candidates["Kumaş tipi"]
    .apply(normalize_fabric_structure)
)

# Prepare SAP structure + weight representation
sap_structure_weight = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .merge(
        sap_structure_candidates[
            ["material_id", "structure"]
        ],
        on="material_id",
        how="inner"
    )
    .merge(
        sap_weight_audit[
            [
                "material_id",
                "sap_weight_raw",
                "sap_weight_unit"
            ]
        ],
        on="material_id",
        how="left"
    )
)

sap_structure_weight = sap_structure_weight[
    sap_structure_weight["sap_weight_raw"].notna()
    & (sap_structure_weight["sap_weight_raw"] > 0)
].copy()


In [ ]:
# Prepare PLM structure + weight representation
plm_structure_weight = (
    plm_structure_candidates[
        [
            "plm_code",
            "material_group_code",
            "structure"
        ]
    ]
    .merge(
        plm_weight_audit[
            [
                "plm_code",
                "plm_weight",
                "plm_weight_unit"
            ]
        ],
        on="plm_code",
        how="left"
    )
)

plm_structure_weight = plm_structure_weight[
    plm_structure_weight["plm_weight"].notna()
    & (plm_structure_weight["plm_weight"] > 0)
].copy()

# Build a fast lookup index
plm_structure_index = {
    key: group.reset_index(drop=True)
    for key, group in (
        plm_structure_weight
        .groupby(
            [
                "material_group_code",
                "structure"
            ],
            dropna=False
        )
    )
}

# Generate candidates that agree on structure
# and are within a weight tolerance
def generate_structure_weight_candidates(
    sap_df,
    plm_index,
    weight_tolerance=0.10
):
    records = []

    for row in sap_df.itertuples(index=False):

        key = (
            row.material_group_code,
            row.structure
        )

        plm_pool = plm_index.get(key)

        if plm_pool is None or plm_pool.empty:
            continue

        sap_weight = row.sap_weight_raw
        sap_unit = row.sap_weight_unit

        plm_weights = (
            plm_pool["plm_weight"]
            .astype(float)
            .to_numpy()
        )

        # Test both observed SAP scale patterns
        same_scale_error = (
            np.abs(sap_weight - plm_weights)
            / plm_weights
        )

        x1000_error = (
            np.abs((sap_weight / 1000) - plm_weights)
            / plm_weights
        )

        best_error = np.minimum(
            same_scale_error,
            x1000_error
        )

        valid = best_error <= weight_tolerance

        # If both units exist, require agreement
        if pd.notna(sap_unit):

            plm_units = (
                plm_pool["plm_weight_unit"]
                .astype("string")
            )

            unit_valid = (
                plm_units.isna()
                | (plm_units == str(sap_unit))
            ).to_numpy()

            valid = valid & unit_valid

        candidate_indices = np.where(valid)[0]

        for idx in candidate_indices:

            records.append(
                (
                    row.material_id,
                    row.material_group_code,
                    plm_pool.iloc[idx]["plm_code"],
                    best_error[idx]
                )
            )

    return pd.DataFrame(
        records,
        columns=[
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "weight_relative_error"
        ]
    )


In [ ]:
structure_weight_candidates = (
    generate_structure_weight_candidates(
        sap_structure_weight,
        plm_structure_index,
        weight_tolerance=0.10
    )
)

structure_weight_candidates[
    "candidate_source"
] = "structure_weight"

print(
    "Structure + weight candidate pairs:",
    len(structure_weight_candidates)
)

print(
    "SAP materials covered:",
    structure_weight_candidates[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(structure_weight_candidates)
    / structure_weight_candidates["material_id"].nunique()
)

# Normalize fiber names for retrieval
def normalize_fiber_for_retrieval(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    # Remove leading percentage / numeric composition
    value = re.sub(
        r"^\s*\d+(?:[.,]\d+)?\s*%?\s*",
        "",
        value
    ).strip()

    if not value or re.fullmatch(r"[\d.,]+", value):
        return pd.NA

    mapping = {
        "POLIESTER": "POLYESTER",
        "ELASTHANE": "ELASTANE",
        "SPANDEX": "ELASTANE",
        "POLYAMIDE6": "POLYAMIDE",
        "POLIAMID6": "POLYAMIDE",
        "TENCEL": "LYOCELL",
        "ACETAT": "ACETATE"
    }

    return mapping.get(value, value)

# Prepare SAP fiber-name sets
sap_fiber_candidates = (
    fabric_attributes[
        fabric_attributes["field_name"]
        == "FIBERCONTENTLISTID"
    ][
        ["material_id", "Karakteristik değeri"]
    ]
    .copy()
)

sap_fiber_candidates["fiber_name"] = (
    sap_fiber_candidates["Karakteristik değeri"]
    .apply(normalize_fiber_for_retrieval)
)

sap_fiber_sets = (
    sap_fiber_candidates
    .dropna(subset=["fiber_name"])
    .groupby("material_id")["fiber_name"]
    .apply(lambda x: tuple(sorted(set(x))))
    .rename("fiber_key")
    .reset_index()
)

# Prepare PLM fiber-name sets
plm_fiber_candidates = (
    plm_multi_value_valid[
        plm_multi_value_valid[
            "PLM Karakteristik Tanımı"
        ] == "FIBERCONTENTLISTID"
    ][
        ["plm_code", "PLM Karakteristik Değeri"]
    ]
    .copy()
)

plm_fiber_candidates["fiber_name"] = (
    plm_fiber_candidates[
        "PLM Karakteristik Değeri"
    ]
    .apply(normalize_fiber_for_retrieval)
)

plm_fiber_sets = (
    plm_fiber_candidates
    .dropna(subset=["fiber_name"])
    .groupby("plm_code")["fiber_name"]
    .apply(lambda x: tuple(sorted(set(x))))
    .rename("fiber_key")
    .reset_index()
)


In [ ]:
# SAP fiber + weight representation
sap_fiber_weight = (
    sap_candidates_source[
        ["material_id", "material_group_code"]
    ]
    .merge(
        sap_fiber_sets,
        on="material_id",
        how="inner"
    )
    .merge(
        sap_weight_audit[
            [
                "material_id",
                "sap_weight_raw",
                "sap_weight_unit"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

sap_fiber_weight = sap_fiber_weight[
    sap_fiber_weight["sap_weight_raw"].notna()
    & (sap_fiber_weight["sap_weight_raw"] > 0)
].copy()

# PLM fiber + weight representation
plm_fiber_weight = (
    plm_codes[
        ["plm_code", "material_group_code"]
    ]
    .merge(
        plm_fiber_sets,
        on="plm_code",
        how="inner"
    )
    .merge(
        plm_weight_audit[
            [
                "plm_code",
                "plm_weight",
                "plm_weight_unit"
            ]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_fiber_weight = plm_fiber_weight[
    plm_fiber_weight["plm_weight"].notna()
    & (plm_fiber_weight["plm_weight"] > 0)
].copy()

# Build PLM lookup by material group and fiber set
plm_fiber_index = {
    key: group.reset_index(drop=True)
    for key, group in (
        plm_fiber_weight
        .groupby(
            ["material_group_code", "fiber_key"],
            dropna=False
        )
    )
}


In [ ]:
# Generate exact-fiber + weight candidates
def generate_fiber_weight_candidates(
    sap_df,
    plm_index,
    weight_tolerance=0.10
):
    records = []

    for row in sap_df.itertuples(index=False):

        key = (
            row.material_group_code,
            row.fiber_key
        )

        plm_pool = plm_index.get(key)

        if plm_pool is None or plm_pool.empty:
            continue

        plm_weights = (
            plm_pool["plm_weight"]
            .astype(float)
            .to_numpy()
        )

        same_scale_error = (
            np.abs(row.sap_weight_raw - plm_weights)
            / plm_weights
        )

        x1000_error = (
            np.abs(
                (row.sap_weight_raw / 1000)
                - plm_weights
            )
            / plm_weights
        )

        best_error = np.minimum(
            same_scale_error,
            x1000_error
        )

        valid = best_error <= weight_tolerance

        # Only enforce unit when SAP unit is known
        if pd.notna(row.sap_weight_unit):

            plm_units = (
                plm_pool["plm_weight_unit"]
                .astype("string")
            )

            unit_valid = (
                plm_units.isna()
                | (
                    plm_units
                    == str(row.sap_weight_unit)
                )
            ).to_numpy()

            valid = valid & unit_valid

        for idx in np.where(valid)[0]:

            records.append({
                "material_id": row.material_id,
                "material_group_code":
                    row.material_group_code,
                "candidate_plm_code":
                    plm_pool.iloc[idx]["plm_code"],
                "weight_relative_error":
                    best_error[idx],
                "candidate_source":
                    "fiber_weight"
            })

    return pd.DataFrame(records)

fiber_weight_candidates = (
    generate_fiber_weight_candidates(
        sap_fiber_weight,
        plm_fiber_index,
        weight_tolerance=0.10
    )
)

print(
    "Fiber + weight candidate pairs:",
    len(fiber_weight_candidates)
)

print(
    "SAP materials covered:",
    fiber_weight_candidates[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(fiber_weight_candidates)
    / fiber_weight_candidates["material_id"].nunique()
)


In [ ]:
# Union of text, structure-weight and fiber-weight channels
all_candidate_pairs = (
    pd.concat(
        [
            tfidf_candidates[
                ["material_id", "candidate_plm_code"]
            ],
            structure_weight_candidates[
                ["material_id", "candidate_plm_code"]
            ],
            fiber_weight_candidates[
                ["material_id", "candidate_plm_code"]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

all_candidate_recall = (
    known_retrieval_pairs
    .merge(
        all_candidate_pairs,
        left_on=["material_id", "plm_code"],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

all_candidate_recall["retrieved"] = (
    all_candidate_recall["_merge"] == "both"
)

print(
    "Combined candidate recall:",
    f"{all_candidate_recall['retrieved'].mean():.2%}"
)

# Evaluate final candidate union by SAP attribute availability
final_candidate_recall = (
    known_retrieval_pairs
    .merge(
        all_candidate_pairs,
        left_on=["material_id", "plm_code"],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left",
        indicator=True
    )
)

final_candidate_recall["retrieved"] = (
    final_candidate_recall["_merge"] == "both"
)

final_candidate_recall["has_sap_attributes"] = (
    final_candidate_recall["material_id"]
    .astype("string")
    .isin(materials_with_attributes)
)

display(
    final_candidate_recall
    .groupby("has_sap_attributes")
    .agg(
        known_pairs=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

# Evaluate attribute-rich candidate recall by material group
final_candidate_recall_grouped = (
    final_candidate_recall
    .merge(
        known_material_groups_model[
            [
                "material_id",
                "material_group_name"
            ]
        ].drop_duplicates("material_id"),
        on="material_id",
        how="left"
    )
)

display(
    final_candidate_recall_grouped[
        final_candidate_recall_grouped[
            "has_sap_attributes"
        ]
    ]
    .groupby("material_group_name")
    .agg(
        known_pairs=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

# Candidate pool size after all retrieval channels
candidate_pool_size = (
    all_candidate_pairs
    .groupby("material_id")
    .size()
    .rename("candidate_count")
)

print(
    candidate_pool_size.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

attribute_candidate_pool_size = (
    candidate_pool_size[
        candidate_pool_size.index.astype("string")
        .isin(materials_with_attributes)
    ]
)

print(
    attribute_candidate_pool_size.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


## 5. Pairwise reranking for materials with structured attributes

Each SAP–PLM candidate pair is reduced to one row before modeling. Pairwise features combine retrieval evidence with fiber, weight, structure, coloring, weave, and yarn signals. Hard negatives are sampled per SAP material, while evaluation is performed on the full candidate pool.


In [ ]:
# Shared lookups used by the clean one-row-per-pair reranker.
material_group_lookup = (
    sap_candidates_source[["material_id", "material_group_code"]]
    .drop_duplicates("material_id")
)

known_attribute_mappings = (
    known_retrieval_pairs[
        known_retrieval_pairs["material_id"].astype("string").isin(materials_with_attributes)
    ][["material_id", "plm_code", "material_group_code"]]
    .drop_duplicates()
    .rename(columns={"plm_code": "true_plm_code"})
    .copy()
)

known_attribute_mappings["material_id"] = known_attribute_mappings["material_id"].astype("string")
known_attribute_mappings["material_group_code"] = known_attribute_mappings["material_group_code"].astype("string")


def tuple_jaccard(left, right):
    if not isinstance(left, tuple) or not isinstance(right, tuple):
        return np.nan
    left_set, right_set = set(left), set(right)
    if not left_set or not right_set:
        return np.nan
    return len(left_set & right_set) / len(left_set | right_set)


# Canonical coloring sets for scalar lookup.
sap_coloring_model = sap_coloring_sets.copy()
plm_coloring_model = plm_coloring_sets.copy()
sap_coloring_model["sap_coloring_key"] = (
    sap_coloring_model["sap_coloring_set"].apply(lambda values: tuple(sorted(values)))
)
plm_coloring_model["plm_coloring_key"] = (
    plm_coloring_model["plm_coloring_set"].apply(lambda values: tuple(sorted(values)))
)

# Aggregate text channel to one row per SAP-PLM pair
text_channel_clean = (
    tfidf_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        retrieval_similarity=(
            "retrieval_similarity",
            "max"
        )
    )
)

text_channel_clean["from_text"] = 1

# Aggregate structure-weight channel
structure_channel_clean = (
    structure_weight_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        structure_weight_error=(
            "weight_relative_error",
            "min"
        )
    )
)

structure_channel_clean[
    "from_structure_weight"
] = 1

# Aggregate fiber-weight channel
fiber_channel_clean = (
    fiber_weight_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        fiber_weight_error=(
            "weight_relative_error",
            "min"
        )
    )
)

fiber_channel_clean[
    "from_fiber_weight"
] = 1


In [ ]:
# Rebuild a strictly one-row-per-pair candidate master
candidate_master_clean = (
    all_candidate_pairs[
        ["material_id", "candidate_plm_code"]
    ]
    .drop_duplicates()
    .merge(
        text_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        structure_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        fiber_channel_clean,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        material_group_lookup,
        on="material_id",
        how="left",
        validate="many_to_one"
    )
)

for column in [
    "from_text",
    "from_structure_weight",
    "from_fiber_weight"
]:
    candidate_master_clean[column] = (
        candidate_master_clean[column]
        .fillna(0)
        .astype(int)
    )

candidate_master_clean["retrieval_channel_count"] = (
    candidate_master_clean[
        [
            "from_text",
            "from_structure_weight",
            "from_fiber_weight"
        ]
    ].sum(axis=1)
)

# Candidate pair must be unique
assert not candidate_master_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

print(
    "Clean candidate pairs:",
    len(candidate_master_clean)
)

# Define the unique pair key
pair_key = [
    "material_id",
    "candidate_plm_code"
]

assert not candidate_master_clean.duplicated(pair_key).any()

# Rebuild labeled candidate pairs from the clean candidate master
pairwise_dataset_clean = (
    candidate_master_clean
    .merge(
        known_attribute_mappings,
        on=[
            "material_id",
            "material_group_code"
        ],
        how="inner",
        validate="many_to_one"
    )
)

pairwise_dataset_clean["is_match"] = (
    pairwise_dataset_clean["candidate_plm_code"]
    == pairwise_dataset_clean["true_plm_code"]
).astype(int)

print(
    "Pairwise rows:",
    len(pairwise_dataset_clean)
)

print(
    "Unique pairs:",
    pairwise_dataset_clean[pair_key]
    .drop_duplicates()
    .shape[0]
)

assert not pairwise_dataset_clean.duplicated(pair_key).any()


In [ ]:
# Keep only materials with a positive candidate
retrieved_material_ids = set(
    pairwise_dataset_clean.loc[
        pairwise_dataset_clean["is_match"] == 1,
        "material_id"
    ]
)

pairwise_training_clean = (
    pairwise_dataset_clean[
        pairwise_dataset_clean["material_id"]
        .isin(retrieved_material_ids)
    ]
    .copy()
)

print(
    "Training materials:",
    pairwise_training_clean[
        "material_id"
    ].nunique()
)

print(
    "Positive pairs:",
    pairwise_training_clean[
        "is_match"
    ].sum()
)

print(
    "Duplicate pairs:",
    pairwise_training_clean
    .duplicated(pair_key)
    .sum()
)

# Build scalar lookup dictionaries
sap_structure_lookup = (
    sap_structure_candidates
    .drop_duplicates("material_id")
    .set_index("material_id")["structure"]
    .to_dict()
)

plm_structure_lookup = (
    plm_structure_candidates
    .drop_duplicates("plm_code")
    .set_index("plm_code")["structure"]
    .to_dict()
)

sap_weight_raw_lookup = (
    sap_weight_audit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weight_raw"]
    .to_dict()
)

sap_weight_unit_lookup = (
    sap_weight_audit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weight_unit"]
    .to_dict()
)

plm_weight_value_lookup = (
    plm_weight_audit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weight"]
    .to_dict()
)

plm_weight_unit_lookup = (
    plm_weight_audit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weight_unit"]
    .to_dict()
)

sap_fiber_lookup = (
    sap_fiber_sets
    .set_index("material_id")["fiber_key"]
    .to_dict()
)

plm_fiber_lookup = (
    plm_fiber_sets
    .set_index("plm_code")["fiber_key"]
    .to_dict()
)

sap_coloring_lookup = (
    sap_coloring_model
    .set_index("material_id")["sap_coloring_key"]
    .to_dict()
)

plm_coloring_lookup = (
    plm_coloring_model
    .set_index("plm_code")["plm_coloring_key"]
    .to_dict()
)

sap_weave_lookup = (
    sap_weave_type
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weave_type"]
    .to_dict()
)

plm_weave_lookup = (
    plm_weave_type
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weave_type"]
    .to_dict()
)


In [ ]:
# Fiber similarity
pairwise_training_clean["fiber_similarity"] = [
    tuple_jaccard(
        sap_fiber_lookup.get(material_id),
        plm_fiber_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Fabric structure match
pairwise_training_clean["fabric_structure_match"] = [
    (
        float(
            sap_structure_lookup.get(material_id)
            == plm_structure_lookup.get(plm_code)
        )
        if (
            sap_structure_lookup.get(material_id) is not None
            and plm_structure_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Coloring match
pairwise_training_clean["coloring_match"] = [
    (
        float(
            sap_coloring_lookup.get(material_id)
            == plm_coloring_lookup.get(plm_code)
        )
        if (
            sap_coloring_lookup.get(material_id) is not None
            and plm_coloring_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]

# Weave type match
pairwise_training_clean["weave_type_match"] = [
    (
        float(
            sap_weave_lookup.get(material_id)
            == plm_weave_lookup.get(plm_code)
        )
        if (
            sap_weave_lookup.get(material_id) is not None
            and plm_weave_lookup.get(plm_code) is not None
        )
        else np.nan
    )
    for material_id, plm_code in zip(
        pairwise_training_clean["material_id"],
        pairwise_training_clean["candidate_plm_code"]
    )
]


In [ ]:
# Map weight values without changing row count
pairwise_training_clean["sap_weight_raw"] = (
    pairwise_training_clean["material_id"]
    .map(sap_weight_raw_lookup)
)

pairwise_training_clean["sap_weight_unit"] = (
    pairwise_training_clean["material_id"]
    .map(sap_weight_unit_lookup)
)

pairwise_training_clean["plm_weight"] = (
    pairwise_training_clean["candidate_plm_code"]
    .map(plm_weight_value_lookup)
)

pairwise_training_clean["plm_weight_unit"] = (
    pairwise_training_clean["candidate_plm_code"]
    .map(plm_weight_unit_lookup)
)

valid_weight = (
    pairwise_training_clean["sap_weight_raw"].notna()
    & pairwise_training_clean["plm_weight"].notna()
    & (pairwise_training_clean["sap_weight_raw"] > 0)
    & (pairwise_training_clean["plm_weight"] > 0)
)

same_scale_error = (
    abs(
        pairwise_training_clean["sap_weight_raw"]
        - pairwise_training_clean["plm_weight"]
    )
    / pairwise_training_clean["plm_weight"]
)

x1000_error = (
    abs(
        pairwise_training_clean["sap_weight_raw"] / 1000
        - pairwise_training_clean["plm_weight"]
    )
    / pairwise_training_clean["plm_weight"]
)

weight_error = np.minimum(
    same_scale_error,
    x1000_error
)

unit_mismatch = (
    pairwise_training_clean["sap_weight_unit"].notna()
    & pairwise_training_clean["plm_weight_unit"].notna()
    & (
        pairwise_training_clean["sap_weight_unit"]
        != pairwise_training_clean["plm_weight_unit"]
    )
)

weight_error[
    ~valid_weight | unit_mismatch
] = np.nan

pairwise_training_clean["weight_similarity"] = (
    1 / (1 + weight_error)
)

assert not pairwise_training_clean.duplicated(
    pair_key
).any()

print(
    "Clean pairwise rows:",
    len(pairwise_training_clean)
)

print(
    "Clean unique pairs:",
    pairwise_training_clean[
        pair_key
    ].drop_duplicates().shape[0]
)

# Final integrity checks
print(
    "Training materials:",
    pairwise_training_clean["material_id"].nunique()
)

print(
    "Positive pairs:",
    pairwise_training_clean["is_match"].sum()
)

print(
    "Materials without a positive:",
    (
        pairwise_training_clean
        .groupby("material_id")["is_match"]
        .sum()
        .eq(0)
        .sum()
    )
)

assert not pairwise_training_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()


In [ ]:
# Build yarn-count lookup dictionaries
sap_knit_yarn_count_lookup = (
    sap_yarn_count_knit
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_yarn_count"]
    .to_dict()
)

plm_knit_yarn_count_lookup = (
    plm_yarn_count_knit
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_yarn_count"]
    .to_dict()
)

sap_weft_yarn_count_lookup = (
    sap_weft_yarn_count
    .drop_duplicates("material_id")
    .set_index("material_id")["sap_weft_yarn_count"]
    .to_dict()
)

plm_weft_yarn_count_lookup = (
    plm_weft_yarn_count
    .drop_duplicates("plm_code")
    .set_index("plm_code")["plm_weft_yarn_count"]
    .to_dict()
)

# Build normalized knitting yarn-type lookups
sap_knit_yarn_type_lookup = (
    sap_yarn_type_sets_normalized
    .set_index("material_id")["sap_yarn_type_set"]
    .to_dict()
)

plm_knit_yarn_type_lookup = (
    plm_yarn_type_sets_normalized
    .set_index("plm_code")["plm_yarn_type_set"]
    .to_dict()
)


def safe_set_jaccard(left, right):
    if not isinstance(left, set) or not isinstance(right, set):
        return np.nan

    if not left or not right:
        return np.nan

    return len(left & right) / len(left | right)

pairwise_training_clean[
    "knit_yarn_count_match"
] = np.nan

pairwise_training_clean[
    "knit_yarn_type_similarity"
] = np.nan

pairwise_training_clean[
    "weft_yarn_count_match"
] = np.nan

orme_mask = (
    pairwise_training_clean["material_group_code"]
    == "1020001"
)

pairwise_training_clean.loc[
    orme_mask,
    "knit_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_knit_yarn_count_lookup.get(material_id),
        plm_knit_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            orme_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            orme_mask,
            "candidate_plm_code"
        ]
    )
]

pairwise_training_clean.loc[
    orme_mask,
    "knit_yarn_type_similarity"
] = [
    safe_set_jaccard(
        sap_knit_yarn_type_lookup.get(material_id),
        plm_knit_yarn_type_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            orme_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            orme_mask,
            "candidate_plm_code"
        ]
    )
]


In [ ]:
dokuma_mask = (
    pairwise_training_clean["material_group_code"]
    == "1020002"
)

pairwise_training_clean.loc[
    dokuma_mask,
    "weft_yarn_count_match"
] = [
    calculate_yarn_count_match(
        sap_weft_yarn_count_lookup.get(material_id),
        plm_weft_yarn_count_lookup.get(plm_code)
    )
    for material_id, plm_code in zip(
        pairwise_training_clean.loc[
            dokuma_mask,
            "material_id"
        ],
        pairwise_training_clean.loc[
            dokuma_mask,
            "candidate_plm_code"
        ]
    )
]

assert not pairwise_training_clean.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

# Heuristic score used only for negative sampling
pairwise_training_clean["hardness_score"] = (
    pairwise_training_clean[
        "fiber_similarity"
    ].fillna(0)
    + pairwise_training_clean[
        "weight_similarity"
    ].fillna(0)
    + pairwise_training_clean[
        "fabric_structure_match"
    ].fillna(0)
    + 0.5
    * pairwise_training_clean[
        "coloring_match"
    ].fillna(0)
    + 0.25
    * pairwise_training_clean[
        "retrieval_similarity"
    ].fillna(0)
)

positive_pairs_clean = (
    pairwise_training_clean[
        pairwise_training_clean["is_match"] == 1
    ]
    .copy()
)

negative_pairs_clean = (
    pairwise_training_clean[
        pairwise_training_clean["is_match"] == 0
    ]
    .copy()
)

hard_negatives_clean = (
    negative_pairs_clean
    .sort_values(
        ["material_id", "hardness_score"],
        ascending=[True, False]
    )
    .groupby("material_id")
    .head(20)
)

rng = np.random.default_rng(42)

negative_pairs_clean["random_score"] = (
    rng.random(len(negative_pairs_clean))
)

random_negatives_clean = (
    negative_pairs_clean
    .sort_values(
        ["material_id", "random_score"]
    )
    .groupby("material_id")
    .head(20)
)

pairwise_sample_clean = (
    pd.concat(
        [
            positive_pairs_clean,
            hard_negatives_clean,
            random_negatives_clean
        ],
        ignore_index=True
    )
    .drop_duplicates(
        [
            "material_id",
            "candidate_plm_code"
        ]
    )
)

print(
    "Sample rows:",
    len(pairwise_sample_clean)
)

print(
    "Materials:",
    pairwise_sample_clean[
        "material_id"
    ].nunique()
)

print(
    "Positive pairs:",
    pairwise_sample_clean[
        "is_match"
    ].sum()
)


In [ ]:
def evaluate_group_models(
    feature_config,
    sample_df,
    full_df,
    train_ids,
    test_ids
):
    results = []

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        train_group = sample_df[
            sample_df["material_id"].isin(train_ids)
            & (
                sample_df["material_group_code"]
                == group_code
            )
        ].copy()

        test_group = full_df[
            full_df["material_id"].isin(test_ids)
            & (
                full_df["material_group_code"]
                == group_code
            )
        ].copy()

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=42
                )
            )
        ])

        model.fit(
            train_group[features],
            train_group["is_match"]
        )

        test_group["score"] = (
            model.predict_proba(
                test_group[features]
            )[:, 1]
        )

        test_group["rank"] = (
            test_group
            .groupby("material_id")["score"]
            .rank(
                ascending=False,
                method="first"
            )
        )

        true_rows = (
            test_group[
                test_group["is_match"] == 1
            ]
        )

        results.append({
            "material_group":
                group_name_mapping[group_code],
            "test_materials":
                len(true_rows),
            "top_1":
                (true_rows["rank"] <= 1).mean(),
            "top_3":
                (true_rows["rank"] <= 3).mean(),
            "top_5":
                (true_rows["rank"] <= 5).mean(),
            "top_10":
                (true_rows["rank"] <= 10).mean(),
            "mrr":
                (1 / true_rows["rank"]).mean()
        })

    return pd.DataFrame(results)



In [ ]:
def evaluate_random_forest_models(
    feature_config,
    sample_df,
    full_df,
    train_ids,
    test_ids
):
    results = []
    models = {}

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        train_group = sample_df[
            sample_df["material_id"].isin(train_ids)
            & (
                sample_df["material_group_code"]
                == group_code
            )
        ].copy()

        test_group = full_df[
            full_df["material_id"].isin(test_ids)
            & (
                full_df["material_group_code"]
                == group_code
            )
        ].copy()

        model = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value=-1,
                    add_indicator=True
                )
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=12,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=42
                )
            )
        ])

        model.fit(
            train_group[features],
            train_group["is_match"]
        )

        test_group["score"] = (
            model.predict_proba(
                test_group[features]
            )[:, 1]
        )

        test_group["rank"] = (
            test_group
            .groupby("material_id")["score"]
            .rank(
                ascending=False,
                method="first"
            )
        )

        true_rows = (
            test_group[
                test_group["is_match"] == 1
            ]
            .copy()
        )

        results.append({
            "material_group":
                group_name_mapping[group_code],
            "test_materials":
                len(true_rows),
            "top_1":
                (true_rows["rank"] <= 1).mean(),
            "top_3":
                (true_rows["rank"] <= 3).mean(),
            "top_5":
                (true_rows["rank"] <= 5).mean(),
            "top_10":
                (true_rows["rank"] <= 10).mean(),
            "mrr":
                (1 / true_rows["rank"]).mean()
        })

        models[group_code] = model

    return pd.DataFrame(results), models


## 6. Train / validation / holdout evaluation

Data is split at the **SAP material level** so candidate pairs from the same material cannot leak across splits. Model selection and confidence thresholds use development data only; the holdout set remains untouched until final evaluation.


In [ ]:
material_table = (
    pairwise_training_clean[
        ["material_id", "material_group_code"]
    ]
    .drop_duplicates("material_id")
)

# First reserve 20% as untouched holdout test
development_materials, holdout_materials = train_test_split(
    material_table,
    test_size=0.20,
    random_state=123,
    stratify=material_table["material_group_code"]
)

# Split development portion into train and validation
train_materials, validation_materials = train_test_split(
    development_materials,
    test_size=0.20,
    random_state=123,
    stratify=development_materials["material_group_code"]
)

train_ids_final = set(train_materials["material_id"])
validation_ids_final = set(validation_materials["material_id"])
holdout_ids = set(holdout_materials["material_id"])

print("Train materials:", len(train_ids_final))
print("Validation materials:", len(validation_ids_final))
print("Holdout materials:", len(holdout_ids))

for name, df in {
    "Train": train_materials,
    "Validation": validation_materials,
    "Holdout": holdout_materials
}.items():
    print(f"\n{name}")
    print(
        df["material_group_code"]
        .value_counts()
        .sort_index()
    )

def build_training_sample(
    full_df,
    train_ids,
    hard_negatives_per_material=20,
    random_negatives_per_material=20,
    random_state=42
):
    train_pool = full_df[
        full_df["material_id"].isin(train_ids)
    ].copy()

    positive_pairs = train_pool[
        train_pool["is_match"] == 1
    ].copy()

    negative_pairs = train_pool[
        train_pool["is_match"] == 0
    ].copy()

    # Heuristic used only to identify difficult negatives
    negative_pairs["hardness_score"] = (
        negative_pairs["fiber_similarity"].fillna(0)
        + negative_pairs["weight_similarity"].fillna(0)
        + negative_pairs["fabric_structure_match"].fillna(0)
        + 0.5 * negative_pairs["coloring_match"].fillna(0)
        + 0.25 * negative_pairs["retrieval_similarity"].fillna(0)
    )

    hard_negatives = (
        negative_pairs
        .sort_values(
            ["material_id", "hardness_score"],
            ascending=[True, False]
        )
        .groupby("material_id")
        .head(hard_negatives_per_material)
    )

    rng = np.random.default_rng(random_state)

    negative_pairs["random_score"] = (
        rng.random(len(negative_pairs))
    )

    random_negatives = (
        negative_pairs
        .sort_values(
            ["material_id", "random_score"]
        )
        .groupby("material_id")
        .head(random_negatives_per_material)
    )

    training_sample = (
        pd.concat(
            [
                positive_pairs,
                hard_negatives,
                random_negatives
            ],
            ignore_index=True
        )
        .drop_duplicates(
            ["material_id", "candidate_plm_code"]
        )
    )

    return training_sample


In [ ]:
training_sample_final = build_training_sample(
    pairwise_training_clean,
    train_ids_final
)

print(
    "Training sample rows:",
    len(training_sample_final)
)

print(
    "Training materials:",
    training_sample_final["material_id"].nunique()
)

print(
    "Positive pairs:",
    training_sample_final["is_match"].sum()
)

print(
    "Positive rate:",
    f"{training_sample_final['is_match'].mean():.2%}"
)

common_features = [
    "retrieval_similarity",
    "from_structure_weight",
    "from_fiber_weight",
    "fiber_similarity",
    "weight_similarity",
    "fabric_structure_match",
    "coloring_match"
]

final_group_features = {
    # ORME
    "1020001": common_features + [
        "knit_yarn_count_match"
    ],

    # DOKUMA
    "1020002": common_features + [
        "weave_type_match",
        "weft_yarn_count_match"
    ],

    # DENIM
    "1030004": common_features + [
        "weave_type_match"
    ]
}

logistic_validation_results = evaluate_group_models(
    final_group_features,
    training_sample_final,
    pairwise_training_clean,
    train_ids_final,
    validation_ids_final
)

display(logistic_validation_results)

rf_validation_results, rf_validation_models = (
    evaluate_random_forest_models(
        final_group_features,
        training_sample_final,
        pairwise_training_clean,
        train_ids_final,
        validation_ids_final
    )
)

display(rf_validation_results)

validation_model_comparison = (
    logistic_validation_results
    .merge(
        rf_validation_results,
        on=[
            "material_group",
            "test_materials"
        ],
        suffixes=(
            "_logistic",
            "_random_forest"
        )
    )
)

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    validation_model_comparison[
        f"{metric}_delta"
    ] = (
        validation_model_comparison[
            f"{metric}_random_forest"
        ]
        - validation_model_comparison[
            f"{metric}_logistic"
        ]
    )

display(validation_model_comparison)

development_ids = set(
    development_materials["material_id"]
)

development_pairwise = (
    pairwise_training_clean[
        pairwise_training_clean["material_id"]
        .isin(development_ids)
    ]
    .copy()
)


In [ ]:
def cross_validate_rerankers(
    full_df,
    feature_config,
    n_splits=5,
    random_state=42
):
    cv_results = []

    group_name_mapping = {
        "1020001": "ORME",
        "1020002": "DOKUMA",
        "1030004": "DENIM"
    }

    for group_code, features in feature_config.items():

        material_ids = (
            full_df.loc[
                full_df["material_group_code"] == group_code,
                "material_id"
            ]
            .drop_duplicates()
            .to_numpy()
        )

        kfold = KFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state
        )

        for fold, (train_idx, val_idx) in enumerate(
            kfold.split(material_ids),
            start=1
        ):
            fold_train_ids = set(
                material_ids[train_idx]
            )

            fold_val_ids = set(
                material_ids[val_idx]
            )

            # Negative sampling only from fold training materials
            training_sample = build_training_sample(
                full_df,
                fold_train_ids,
                random_state=42 + fold
            )

            validation_full = (
                full_df[
                    full_df["material_id"]
                    .isin(fold_val_ids)
                ]
                .copy()
            )

            models = {
                "logistic": Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="constant",
                            fill_value=-1,
                            add_indicator=True
                        )
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    ),
                    (
                        "model",
                        LogisticRegression(
                            max_iter=2000,
                            class_weight="balanced",
                            random_state=42
                        )
                    )
                ]),

                "random_forest":
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value=-1,
                                add_indicator=True
                            )
                        ),
                        (
                            "model",
                            RandomForestClassifier(
                                n_estimators=300,
                                max_depth=12,
                                min_samples_leaf=2,
                                class_weight=
                                    "balanced_subsample",
                                n_jobs=-1,
                                random_state=42
                            )
                        )
                    ])
            }

            for model_name, model in models.items():

                model.fit(
                    training_sample[features],
                    training_sample["is_match"]
                )

                scored = validation_full.copy()

                scored["score"] = (
                    model.predict_proba(
                        scored[features]
                    )[:, 1]
                )

                scored["rank"] = (
                    scored
                    .groupby("material_id")["score"]
                    .rank(
                        ascending=False,
                        method="first"
                    )
                )

                true_rows = (
                    scored[
                        scored["is_match"] == 1
                    ]
                )

                cv_results.append({
                    "material_group":
                        group_name_mapping[group_code],
                    "fold": fold,
                    "model": model_name,
                    "test_materials":
                        len(true_rows),

                    "top_1":
                        (
                            true_rows["rank"] <= 1
                        ).mean(),

                    "top_3":
                        (
                            true_rows["rank"] <= 3
                        ).mean(),

                    "top_5":
                        (
                            true_rows["rank"] <= 5
                        ).mean(),

                    "top_10":
                        (
                            true_rows["rank"] <= 10
                        ).mean(),

                    "mrr":
                        (
                            1 / true_rows["rank"]
                        ).mean()
                })

    return pd.DataFrame(cv_results)


In [ ]:
cv_results = cross_validate_rerankers(
    development_pairwise,
    final_group_features,
    n_splits=5,
    random_state=42
)

cv_summary = (
    cv_results
    .groupby(
        ["material_group", "model"]
    )
    .agg(
        folds=("fold", "count"),
        top_1_mean=("top_1", "mean"),
        top_1_std=("top_1", "std"),
        top_3_mean=("top_3", "mean"),
        top_5_mean=("top_5", "mean"),
        top_10_mean=("top_10", "mean"),
        mrr_mean=("mrr", "mean"),
        mrr_std=("mrr", "std")
    )
    .reset_index()
)

display(cv_summary)

cv_comparison = (
    cv_summary
    .pivot(
        index="material_group",
        columns="model",
        values=[
            "top_1_mean",
            "top_5_mean",
            "mrr_mean"
        ]
    )
)

display(cv_comparison)

# Combine train and validation for final development training
development_ids_final = (
    train_ids_final
    | validation_ids_final
)

final_training_sample = build_training_sample(
    pairwise_training_clean,
    development_ids_final,
    hard_negatives_per_material=20,
    random_negatives_per_material=20,
    random_state=42
)

print(
    "Development training materials:",
    final_training_sample["material_id"].nunique()
)

print(
    "Training sample rows:",
    len(final_training_sample)
)

print(
    "Positive pairs:",
    final_training_sample["is_match"].sum()
)


In [ ]:
final_rf_models = {}
holdout_results = []
holdout_scored_parts = []

group_name_mapping = {
    "1020001": "ORME",
    "1020002": "DOKUMA",
    "1030004": "DENIM"
}

for group_code, features in final_group_features.items():

    train_group = final_training_sample[
        final_training_sample["material_group_code"]
        == group_code
    ].copy()

    holdout_group = pairwise_training_clean[
        pairwise_training_clean["material_id"].isin(holdout_ids)
        & (
            pairwise_training_clean["material_group_code"]
            == group_code
        )
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    holdout_group["match_probability"] = (
        model.predict_proba(
            holdout_group[features]
        )[:, 1]
    )

    holdout_group["predicted_rank"] = (
        holdout_group
        .groupby("material_id")["match_probability"]
        .rank(
            ascending=False,
            method="first"
        )
    )

    true_rows = holdout_group[
        holdout_group["is_match"] == 1
    ].copy()

    holdout_results.append({
        "material_group":
            group_name_mapping[group_code],

        "holdout_materials":
            len(true_rows),

        "top_1":
            (true_rows["predicted_rank"] <= 1).mean(),

        "top_3":
            (true_rows["predicted_rank"] <= 3).mean(),

        "top_5":
            (true_rows["predicted_rank"] <= 5).mean(),

        "top_10":
            (true_rows["predicted_rank"] <= 10).mean(),

        "mrr":
            (
                1 / true_rows["predicted_rank"]
            ).mean()
    })

    final_rf_models[group_code] = model
    holdout_scored_parts.append(holdout_group)

holdout_results = pd.DataFrame(
    holdout_results
)

holdout_scored = pd.concat(
    holdout_scored_parts,
    ignore_index=True
)

display(holdout_results)


In [ ]:
total_holdout = (
    holdout_results["holdout_materials"].sum()
)

overall_holdout = {}

for metric in [
    "top_1",
    "top_3",
    "top_5",
    "top_10",
    "mrr"
]:
    overall_holdout[metric] = (
        (
            holdout_results[metric]
            * holdout_results["holdout_materials"]
        ).sum()
        / total_holdout
    )

print("Reranker holdout performance")

for metric, value in overall_holdout.items():
    print(
        f"{metric}: {value:.4f}"
    )

holdout_true_ranks = (
    holdout_scored[
        holdout_scored["is_match"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "true_plm_code",
            "predicted_rank",
            "match_probability"
        ]
    ]
    .copy()
)

display(
    holdout_true_ranks[
        "predicted_rank"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

# Score validation candidates using the selected Random Forest models
validation_scored_parts = []

for group_code, features in final_group_features.items():

    train_group = training_sample_final[
        training_sample_final["material_group_code"]
        == group_code
    ].copy()

    validation_group = pairwise_training_clean[
        pairwise_training_clean["material_id"]
        .isin(validation_ids_final)
        & (
            pairwise_training_clean["material_group_code"]
            == group_code
        )
    ].copy()

    model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    validation_group["score"] = (
        model.predict_proba(
            validation_group[features]
        )[:, 1]
    )

    validation_scored_parts.append(
        validation_group
    )

validation_scored = pd.concat(
    validation_scored_parts,
    ignore_index=True
)


In [ ]:
# Rank validation candidates
validation_scored["rank"] = (
    validation_scored
    .groupby("material_id")["score"]
    .rank(
        ascending=False,
        method="first"
    )
)

top_two = (
    validation_scored[
        validation_scored["rank"] <= 2
    ]
    .sort_values(
        ["material_id", "rank"]
    )
)

confidence_table = (
    top_two
    .pivot(
        index="material_id",
        columns="rank",
        values="score"
    )
    .rename(
        columns={
            1.0: "top1_score",
            2.0: "top2_score"
        }
    )
    .reset_index()
)

confidence_table["score_margin"] = (
    confidence_table["top1_score"]
    - confidence_table["top2_score"]
)

top1_predictions = (
    validation_scored[
        validation_scored["rank"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "true_plm_code",
            "score"
        ]
    ]
    .copy()
)

top1_predictions["is_top1_correct"] = (
    top1_predictions["candidate_plm_code"]
    == top1_predictions["true_plm_code"]
)

confidence_audit = (
    top1_predictions
    .merge(
        confidence_table,
        on="material_id",
        how="left"
    )
)

confidence_audit["margin_band"] = pd.cut(
    confidence_audit["score_margin"],
    bins=[
        -np.inf,
        0.05,
        0.10,
        0.20,
        0.30,
        np.inf
    ],
    labels=[
        "<=0.05",
        "0.05-0.10",
        "0.10-0.20",
        "0.20-0.30",
        ">0.30"
    ]
)

display(
    confidence_audit
    .groupby(
        "margin_band",
        observed=True
    )
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_top1_score=("top1_score", "mean"),
        mean_margin=("score_margin", "mean")
    )
)

display(
    confidence_audit
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_top1_score=("top1_score", "mean"),
        mean_margin=("score_margin", "mean")
    )
)


In [ ]:
# Evaluate cumulative confidence thresholds
margin_thresholds = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50
]

threshold_results = []

total_materials = len(confidence_audit)

for threshold in margin_thresholds:

    selected = confidence_audit[
        confidence_audit["score_margin"] >= threshold
    ]

    if len(selected) == 0:
        continue

    threshold_results.append({
        "margin_threshold": threshold,
        "selected_materials": len(selected),
        "coverage": len(selected) / total_materials,
        "precision": selected["is_top1_correct"].mean()
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(threshold_results)

# Evaluate confidence thresholds by material group
group_threshold_results = []

for group_code, group in confidence_audit.groupby(
    "material_group_code"
):

    group_total = len(group)

    for threshold in margin_thresholds:

        selected = group[
            group["score_margin"] >= threshold
        ]

        if len(selected) == 0:
            continue

        group_threshold_results.append({
            "material_group_code": group_code,
            "margin_threshold": threshold,
            "selected_materials": len(selected),
            "coverage": len(selected) / group_total,
            "precision": selected[
                "is_top1_correct"
            ].mean()
        })

group_threshold_results = pd.DataFrame(
    group_threshold_results
)

display(group_threshold_results)

# Find the highest-coverage threshold
# satisfying a target precision
def find_best_threshold(
    results,
    target_precision
):
    eligible = results[
        results["precision"] >= target_precision
    ].copy()

    if eligible.empty:
        return None

    return (
        eligible
        .sort_values(
            [
                "coverage",
                "margin_threshold"
            ],
            ascending=[False, True]
        )
        .iloc[0]
    )


for target in [0.90, 0.95]:
    result = find_best_threshold(
        threshold_results,
        target
    )

    print(
        f"\nTarget precision: {target:.0%}"
    )

    if result is None:
        print("No threshold reached the target.")
    else:
        print(
            f"Margin threshold: "
            f"{result['margin_threshold']:.2f}"
        )
        print(
            f"Coverage: "
            f"{result['coverage']:.2%}"
        )
        print(
            f"Observed precision: "
            f"{result['precision']:.2%}"
        )


In [ ]:
# Confidence thresholds selected using validation data only
auto_margin_thresholds = {
    "1020001": 0.15,  # ORME
    "1020002": 0.30,  # DOKUMA
    "1030004": 0.40   # DENIM
}

review_margin_threshold = 0.05

# Extract top two candidates from holdout results
holdout_top_two = (
    holdout_scored[
        holdout_scored["predicted_rank"] <= 2
    ]
    .sort_values(
        ["material_id", "predicted_rank"]
    )
)

holdout_confidence = (
    holdout_top_two
    .pivot(
        index="material_id",
        columns="predicted_rank",
        values="match_probability"
    )
    .rename(
        columns={
            1.0: "top1_score",
            2.0: "top2_score"
        }
    )
    .reset_index()
)

holdout_confidence["score_margin"] = (
    holdout_confidence["top1_score"]
    - holdout_confidence["top2_score"]
)

# Add top-1 prediction and correctness
holdout_top1 = (
    holdout_scored[
        holdout_scored["predicted_rank"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "candidate_plm_code",
            "true_plm_code",
            "match_probability"
        ]
    ]
    .copy()
)

holdout_top1["is_top1_correct"] = (
    holdout_top1["candidate_plm_code"]
    == holdout_top1["true_plm_code"]
)

holdout_decisions = (
    holdout_top1
    .merge(
        holdout_confidence,
        on="material_id",
        how="left",
        validate="one_to_one"
    )
)

# Assign operational decision based on fixed validation thresholds
def assign_confidence_decision(row):

    auto_threshold = auto_margin_thresholds[
        row["material_group_code"]
    ]

    if row["score_margin"] >= auto_threshold:
        return "AUTO"

    if row["score_margin"] >= review_margin_threshold:
        return "REVIEW"

    return "LOW_CONFIDENCE"


holdout_decisions["decision"] = (
    holdout_decisions.apply(
        assign_confidence_decision,
        axis=1
    )
)

# Evaluate decision-layer performance
decision_summary = (
    holdout_decisions
    .groupby("decision")
    .agg(
        materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

decision_summary["coverage"] = (
    decision_summary["materials"]
    / len(holdout_decisions)
)

display(decision_summary)


In [ ]:
# Audit automatic recommendations by material group
auto_holdout_summary = (
    holdout_decisions[
        holdout_decisions["decision"] == "AUTO"
    ]
    .groupby("material_group_code")
    .agg(
        auto_materials=("material_id", "size"),
        auto_precision=("is_top1_correct", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

group_holdout_counts = (
    holdout_decisions
    .groupby("material_group_code")
    ["material_id"]
    .size()
    .rename("total_materials")
    .reset_index()
)

auto_holdout_summary = (
    auto_holdout_summary
    .merge(
        group_holdout_counts,
        on="material_group_code",
        how="left"
    )
)

auto_holdout_summary["auto_coverage"] = (
    auto_holdout_summary["auto_materials"]
    / auto_holdout_summary["total_materials"]
)

display(auto_holdout_summary)

# Overall automatic-decision performance
auto_predictions = (
    holdout_decisions[
        holdout_decisions["decision"] == "AUTO"
    ]
)

print(
    "AUTO coverage:",
    f"{len(auto_predictions) / len(holdout_decisions):.2%}"
)

print(
    "AUTO precision:",
    f"{auto_predictions['is_top1_correct'].mean():.2%}"
)

# Add true-PLM presence within Top-K to the holdout decision table
holdout_rank_lookup = (
    holdout_scored[
        holdout_scored["is_match"] == 1
    ][
        [
            "material_id",
            "predicted_rank"
        ]
    ]
    .rename(
        columns={
            "predicted_rank": "true_plm_rank"
        }
    )
)

holdout_decisions = (
    holdout_decisions
    .merge(
        holdout_rank_lookup,
        on="material_id",
        how="left",
        validate="one_to_one"
    )
)

holdout_decisions["true_in_top3"] = (
    holdout_decisions["true_plm_rank"] <= 3
)

holdout_decisions["true_in_top5"] = (
    holdout_decisions["true_plm_rank"] <= 5
)

review_holdout = (
    holdout_decisions[
        holdout_decisions["decision"] == "REVIEW"
    ]
)

print(
    "REVIEW coverage:",
    f"{len(review_holdout) / len(holdout_decisions):.2%}"
)

print(
    "REVIEW Top-1 accuracy:",
    f"{review_holdout['is_top1_correct'].mean():.2%}"
)

print(
    "REVIEW Top-3 success:",
    f"{review_holdout['true_in_top3'].mean():.2%}"
)

print(
    "REVIEW Top-5 success:",
    f"{review_holdout['true_in_top5'].mean():.2%}"
)


In [ ]:
review_group_summary = (
    review_holdout
    .groupby("material_group_code")
    .agg(
        review_materials=("material_id", "size"),
        top1_accuracy=("is_top1_correct", "mean"),
        top3_success=("true_in_top3", "mean"),
        top5_success=("true_in_top5", "mean"),
        mean_margin=("score_margin", "mean")
    )
    .reset_index()
)

display(review_group_summary)

# Operational success under the fixed decision policy
holdout_decisions["workflow_success"] = np.select(
    [
        holdout_decisions["decision"] == "AUTO",
        holdout_decisions["decision"] == "REVIEW"
    ],
    [
        holdout_decisions["is_top1_correct"],
        holdout_decisions["true_in_top3"]
    ],
    default=False
)

actionable = (
    holdout_decisions["decision"]
    .isin(["AUTO", "REVIEW"])
)

print(
    "AUTO + REVIEW coverage:",
    f"{actionable.mean():.2%}"
)

print(
    "Success among actionable materials:",
    f"{holdout_decisions.loc[actionable, 'workflow_success'].mean():.2%}"
)


## 7. Fallback pipeline for SAP materials without structured attributes

Records without SAP characteristics require a separate, lower-confidence path. This branch tests multiple text channels and description-derived pseudo-attributes. Its results should be interpreted separately from the structured-attribute pipeline because candidate recall is the main bottleneck.


In [ ]:
# Prepare known mappings without SAP attributes
known_no_attribute_mappings = (
    known_retrieval_pairs[
        ~known_retrieval_pairs["material_id"]
        .astype("string")
        .isin(materials_with_attributes)
    ]
    .copy()
)

print(
    "Known mappings without SAP attributes:",
    len(known_no_attribute_mappings)
)

# Add the SAP-side material group used for blocking
known_no_attribute_audit = (
    known_no_attribute_mappings
    .merge(
        material_group_lookup.rename(
            columns={
                "material_group_code":
                "sap_material_group_code"
            }
        ),
        on="material_id",
        how="left"
    )
)

known_no_attribute_audit["group_match"] = (
    known_no_attribute_audit["material_group_code"]
    == known_no_attribute_audit["sap_material_group_code"]
)

print(
    "Material-group mismatches:",
    (~known_no_attribute_audit["group_match"]).sum()
)

display(
    known_no_attribute_audit[
        ~known_no_attribute_audit["group_match"]
    ].head(20)
)

no_attribute_materials = (
    known_no_attribute_audit[
        [
            "material_id",
            "sap_material_group_code"
        ]
    ]
    .drop_duplicates("material_id")
    .rename(
        columns={
            "sap_material_group_code":
            "material_group_code"
        }
    )
)

no_attr_development, no_attr_holdout = train_test_split(
    no_attribute_materials,
    test_size=0.20,
    random_state=321,
    stratify=no_attribute_materials[
        "material_group_code"
    ]
)

no_attr_dev_ids = set(
    no_attr_development["material_id"]
)

no_attr_holdout_ids = set(
    no_attr_holdout["material_id"]
)

print(
    "Development materials:",
    len(no_attr_dev_ids)
)

print(
    "Untouched holdout materials:",
    len(no_attr_holdout_ids)
)


In [ ]:
def generate_text_channel_candidates(
    sap_df,
    plm_df,
    material_group_code,
    plm_text_column,
    channel_name,
    top_k=50,
    analyzer="char_wb",
    ngram_range=(3, 5)
):
    sap_group = (
        sap_df[
            sap_df["material_group_code"]
            == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    plm_group = (
        plm_df[
            plm_df["material_group_code"]
            == material_group_code
        ]
        .copy()
        .reset_index(drop=True)
    )

    sap_group = sap_group[
        sap_group["retrieval_text"].str.len() > 0
    ].reset_index(drop=True)

    plm_group = plm_group[
        plm_group[plm_text_column]
        .fillna("")
        .str.len() > 0
    ].reset_index(drop=True)

    if sap_group.empty or plm_group.empty:
        return pd.DataFrame()

    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        sublinear_tf=True,
        norm="l2"
    )

    plm_matrix = vectorizer.fit_transform(
        plm_group[plm_text_column]
    )

    sap_matrix = vectorizer.transform(
        sap_group["retrieval_text"]
    )

    n_neighbors = min(
        top_k,
        len(plm_group)
    )

    model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="cosine",
        algorithm="brute"
    )

    model.fit(plm_matrix)

    distances, indices = model.kneighbors(
        sap_matrix
    )

    records = []

    for sap_idx in range(len(sap_group)):
        for rank, (plm_idx, distance) in enumerate(
            zip(
                indices[sap_idx],
                distances[sap_idx]
            ),
            start=1
        ):
            records.append({
                "material_id":
                    sap_group.loc[
                        sap_idx,
                        "material_id"
                    ],
                "candidate_plm_code":
                    plm_group.loc[
                        plm_idx,
                        "plm_code"
                    ],
                "channel":
                    channel_name,
                "channel_rank":
                    rank,
                "channel_similarity":
                    1 - distance
            })

    return pd.DataFrame(records)

sap_no_attr_dev = (
    sap_candidates_source[
        sap_candidates_source["material_id"]
        .isin(no_attr_dev_ids)
    ]
    .copy()
)


In [ ]:
text_channel_tables = []

channel_config = {
    "original_text": "retrieval_text",
    "structured_text": "structured_text",
    "multi_value_text": "multi_value_text",
    "enriched_text": "retrieval_text_enriched"
}

for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    for channel_name, text_column in (
        channel_config.items()
    ):
        result = generate_text_channel_candidates(
            sap_no_attr_dev,
            plm_candidates_enriched,
            material_group_code=group_code,
            plm_text_column=text_column,
            channel_name=channel_name,
            top_k=50
        )

        if not result.empty:
            text_channel_tables.append(result)

multi_channel_candidates = pd.concat(
    text_channel_tables,
    ignore_index=True
)

multi_channel_union = (
    multi_channel_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Candidate pairs:",
    len(multi_channel_union)
)

print(
    "Average candidates per SAP:",
    len(multi_channel_union)
    / multi_channel_union["material_id"].nunique()
)

no_attr_dev_truth = (
    known_no_attribute_audit[
        known_no_attribute_audit["material_id"]
        .isin(no_attr_dev_ids)
    ][
        [
            "material_id",
            "plm_code"
        ]
    ]
    .copy()
)

no_attr_dev_recall = (
    no_attr_dev_truth
    .merge(
        multi_channel_union,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_dev_recall["retrieved"] = (
    no_attr_dev_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Multi-channel development recall:",
    f"{no_attr_dev_recall['retrieved'].mean():.2%}"
)


In [ ]:
channel_recall_summary = []

for channel_name in (
    multi_channel_candidates["channel"]
    .unique()
):
    channel_candidates = (
        multi_channel_candidates[
            multi_channel_candidates["channel"]
            == channel_name
        ][
            [
                "material_id",
                "candidate_plm_code"
            ]
        ]
        .drop_duplicates()
    )

    evaluation = (
        no_attr_dev_truth
        .merge(
            channel_candidates,
            left_on=[
                "material_id",
                "plm_code"
            ],
            right_on=[
                "material_id",
                "candidate_plm_code"
            ],
            how="left"
        )
    )

    channel_recall_summary.append({
        "channel": channel_name,
        "recall": (
            evaluation[
                "candidate_plm_code"
            ].notna().mean()
        )
    })

display(
    pd.DataFrame(
        channel_recall_summary
    ).sort_values(
        "recall",
        ascending=False
    )
)

# Evaluate multi-channel recall by material group
no_attr_dev_recall_grouped = (
    no_attr_dev_recall
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
)

display(
    no_attr_dev_recall_grouped
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)

# Build training data for description-to-structure prediction
structure_prediction_data = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_structure_candidates[
            [
                "material_id",
                "structure"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

structure_prediction_data = (
    structure_prediction_data[
        structure_prediction_data["retrieval_text"].str.len() > 0
    ]
    .dropna(
        subset=[
            "material_group_code",
            "structure"
        ]
    )
    .copy()
)

display(
    structure_prediction_data
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "nunique"),
        structure_classes=("structure", "nunique")
    )
)



In [ ]:
# Evaluate description-based fabric-structure prediction

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, top_k_accuracy_score

structure_model_results = []
structure_models = {}

TARGET_GROUPS = [
    "1020001",
    "1020002",
    "1030004",
]

for group_code in TARGET_GROUPS:

    # Select data for the current material group
    group_data = (
        structure_prediction_data[
            structure_prediction_data["material_group_code"]
            == group_code
        ]
        .copy()
    )

    # Remove rows with missing target or retrieval text
    group_data = group_data.dropna(
        subset=[
            "structure",
            "retrieval_text",
        ]
    ).copy()

    # Very rare labels are not suitable for stratified classification.
    # Keep only structure classes with at least 2 observations.
    label_counts = (
        group_data["structure"]
        .value_counts()
    )

    valid_labels = label_counts[
        label_counts >= 2
    ].index

    group_data = group_data[
        group_data["structure"].isin(valid_labels)
    ].copy()

    n_samples = len(group_data)
    n_classes = group_data["structure"].nunique()

    print(
        f"\nMaterial group {group_code}: "
        f"{n_samples} materials, "
        f"{n_classes} structure classes"
    )

    # Logistic regression requires at least two target classes
    if n_classes < 2:
        print(
            f"Skipping {group_code}: "
            "fewer than two valid structure classes."
        )
        continue

    # Ensure the test set can contain at least one observation
    # from every class when stratifying.
    test_size = max(
        n_classes,
        int(round(n_samples * 0.20)),
    )

    # The training set must also retain at least one sample per class
    if n_samples - test_size < n_classes:
        print(
            f"Skipping {group_code}: "
            "not enough observations for a stratified "
            "train/test split."
        )
        continue

    train_data, test_data = train_test_split(
        group_data,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=group_data["structure"],
    )

    # Character-level TF-IDF is useful for material descriptions
    # because it captures abbreviations, spelling variants,
    # and recurring technical substrings.
    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                min_df=2,
            ),
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ])

    # Train the classifier
    model.fit(
        train_data["retrieval_text"],
        train_data["structure"],
    )

    # Generate predictions
    predicted = model.predict(
        test_data["retrieval_text"]
    )

    probabilities = model.predict_proba(
        test_data["retrieval_text"]
    )

    classes = model.named_steps[
        "model"
    ].classes_

    # Top-1 accuracy
    top_1_accuracy = accuracy_score(
        test_data["structure"],
        predicted,
    )

    # Top-k accuracy
    # If fewer than 3 classes exist, use the maximum possible k.
    top_k = min(
        3,
        len(classes),
    )

    if top_k >= len(classes):
        # If k equals the number of classes,
        # every true class is necessarily inside the top-k predictions.
        top_3_accuracy = 1.0
    else:
        top_3_accuracy = top_k_accuracy_score(
            test_data["structure"],
            probabilities,
            k=top_k,
            labels=classes,
        )

    structure_model_results.append({
        "material_group_code": group_code,
        "train_materials": len(train_data),
        "test_materials": len(test_data),
        "classes": len(classes),
        "top_1_accuracy": top_1_accuracy,
        "top_3_accuracy": top_3_accuracy,
    })

    # Store the trained model for later use
    structure_models[group_code] = model


# Present evaluation results
structure_model_results_df = pd.DataFrame(
    structure_model_results
)

display(structure_model_results_df)

In [ ]:
# Retrain structure classifiers using all available labeled SAP materials
final_structure_models = {}

for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    group_data = (
        structure_prediction_data[
            structure_prediction_data["material_group_code"]
            == group_code
        ]
        .copy()
    )

    # Keep labels with at least two observations
    label_counts = group_data["structure"].value_counts()

    valid_labels = label_counts[
        label_counts >= 2
    ].index

    group_data = group_data[
        group_data["structure"].isin(valid_labels)
    ].copy()

    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                min_df=2
            )
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])

    model.fit(
        group_data["retrieval_text"],
        group_data["structure"]
    )

    final_structure_models[group_code] = model

# Predict Top-3 fabric structures for SAP materials without attributes
predicted_structure_records = []

for group_code, model in final_structure_models.items():

    sap_group = (
        sap_no_attr_dev[
            sap_no_attr_dev["material_group_code"]
            == group_code
        ]
        .copy()
    )

    if sap_group.empty:
        continue

    probabilities = model.predict_proba(
        sap_group["retrieval_text"]
    )

    classes = model.named_steps[
        "model"
    ].classes_

    top_k = min(3, len(classes))

    top_indices = np.argsort(
        probabilities,
        axis=1
    )[:, -top_k:][:, ::-1]

    for row_idx, material_id in enumerate(
        sap_group["material_id"]
    ):
        for rank, class_idx in enumerate(
            top_indices[row_idx],
            start=1
        ):
            predicted_structure_records.append({
                "material_id": material_id,
                "material_group_code": group_code,
                "predicted_structure":
                    classes[class_idx],
                "structure_rank": rank,
                "structure_probability":
                    probabilities[row_idx, class_idx]
            })

predicted_structures = pd.DataFrame(
    predicted_structure_records
)

display(predicted_structures.head(10))


In [ ]:
# Prepare PLM structure-aware retrieval table
plm_structure_retrieval = (
    plm_candidates_enriched[
        [
            "plm_code",
            "material_group_code",
            "structured_text"
        ]
    ]
    .merge(
        plm_structure_candidates[
            [
                "plm_code",
                "structure"
            ]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_structure_retrieval = (
    plm_structure_retrieval[
        plm_structure_retrieval[
            "structured_text"
        ].str.len() > 0
    ]
    .copy()
)


In [ ]:
def generate_predicted_structure_candidates(
    sap_df,
    predicted_structures,
    plm_df,
    top_k_per_structure=30
):
    candidate_records = []

    for group_code in predicted_structures[
        "material_group_code"
    ].unique():

        sap_group = (
            sap_df[
                sap_df["material_group_code"]
                == group_code
            ]
            .set_index("material_id")
        )

        predictions_group = (
            predicted_structures[
                predicted_structures[
                    "material_group_code"
                ] == group_code
            ]
        )

        for structure_value, structure_predictions in (
            predictions_group.groupby(
                "predicted_structure"
            )
        ):

            plm_pool = (
                plm_df[
                    (plm_df["material_group_code"]
                     == group_code)
                    & (
                        plm_df["structure"]
                        == structure_value
                    )
                ]
                .reset_index(drop=True)
            )

            if plm_pool.empty:
                continue

            vectorizer = TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                norm="l2"
            )

            plm_matrix = vectorizer.fit_transform(
                plm_pool["structured_text"]
            )

            n_neighbors = min(
                top_k_per_structure,
                len(plm_pool)
            )

            nn_model = NearestNeighbors(
                n_neighbors=n_neighbors,
                metric="cosine",
                algorithm="brute"
            )

            nn_model.fit(plm_matrix)

            for pred in structure_predictions.itertuples(
                index=False
            ):

                if pred.material_id not in sap_group.index:
                    continue

                sap_text = sap_group.loc[
                    pred.material_id,
                    "retrieval_text"
                ]

                sap_vector = vectorizer.transform(
                    [sap_text]
                )

                distances, indices = (
                    nn_model.kneighbors(
                        sap_vector
                    )
                )

                for retrieval_rank, (
                    plm_idx,
                    distance
                ) in enumerate(
                    zip(
                        indices[0],
                        distances[0]
                    ),
                    start=1
                ):
                    candidate_records.append({
                        "material_id":
                            pred.material_id,
                        "candidate_plm_code":
                            plm_pool.loc[
                                plm_idx,
                                "plm_code"
                            ],
                        "predicted_structure":
                            structure_value,
                        "structure_rank":
                            pred.structure_rank,
                        "structure_probability":
                            pred.structure_probability,
                        "structure_retrieval_rank":
                            retrieval_rank,
                        "structure_retrieval_similarity":
                            1 - distance
                    })

    return pd.DataFrame(candidate_records)


In [ ]:
pseudo_structure_candidates = (
    generate_predicted_structure_candidates(
        sap_no_attr_dev,
        predicted_structures,
        plm_structure_retrieval,
        top_k_per_structure=30
    )
)

pseudo_structure_pairs = (
    pseudo_structure_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Pseudo-structure candidate pairs:",
    len(pseudo_structure_pairs)
)

print(
    "Average candidates per SAP:",
    len(pseudo_structure_pairs)
    / pseudo_structure_pairs[
        "material_id"
    ].nunique()
)

pseudo_structure_recall = (
    no_attr_dev_truth
    .merge(
        pseudo_structure_pairs,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

pseudo_structure_recall["retrieved"] = (
    pseudo_structure_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Pseudo-structure recall:",
    f"{pseudo_structure_recall['retrieved'].mean():.2%}"
)

# Union text channels with predicted-structure retrieval
no_attr_combined_candidates = (
    pd.concat(
        [
            multi_channel_union,
            pseudo_structure_pairs
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

no_attr_combined_recall = (
    no_attr_dev_truth
    .merge(
        no_attr_combined_candidates,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_combined_recall["retrieved"] = (
    no_attr_combined_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "Combined no-attribute recall:",
    f"{no_attr_combined_recall['retrieved'].mean():.2%}"
)

no_attr_combined_grouped = (
    no_attr_combined_recall
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
)

display(
    no_attr_combined_grouped
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)


In [ ]:
# Convert yarn count into a retrieval-friendly canonical label
def canonicalize_yarn_count(value):
    parsed = parse_yarn_count(value)

    if pd.isna(parsed["count"]):
        return pd.NA

    count = parsed["count"]

    if float(count).is_integer():
        count_text = str(int(count))
    else:
        count_text = str(count)

    ply = parsed["ply"]

    # Missing ply and /1 are treated as equivalent for candidate retrieval
    if pd.isna(ply) or ply == 1:
        return count_text

    if float(ply).is_integer():
        ply_text = str(int(ply))
    else:
        ply_text = str(ply)

    return f"{count_text}/{ply_text}"

# Build description-to-yarn-count training data for knitted fabrics
yarn_count_prediction_data = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_yarn_count_knit[
            [
                "material_id",
                "sap_yarn_count"
            ]
        ],
        on="material_id",
        how="inner"
    )
)

yarn_count_prediction_data = (
    yarn_count_prediction_data[
        yarn_count_prediction_data["material_group_code"]
        == "1020001"
    ]
    .copy()
)

yarn_count_prediction_data["yarn_count_class"] = (
    yarn_count_prediction_data["sap_yarn_count"]
    .apply(canonicalize_yarn_count)
)

yarn_count_prediction_data = (
    yarn_count_prediction_data
    .dropna(subset=["yarn_count_class"])
)

print(
    "ORME materials:",
    len(yarn_count_prediction_data)
)

print(
    "Yarn-count classes:",
    yarn_count_prediction_data[
        "yarn_count_class"
    ].nunique()
)

display(
    yarn_count_prediction_data[
        "yarn_count_class"
    ].value_counts().head(20)
)


In [ ]:
# Keep classes that have enough observations for stratified evaluation
class_counts = (
    yarn_count_prediction_data[
        "yarn_count_class"
    ].value_counts()
)

valid_classes = class_counts[
    class_counts >= 2
].index

yarn_count_model_data = (
    yarn_count_prediction_data[
        yarn_count_prediction_data[
            "yarn_count_class"
        ].isin(valid_classes)
    ]
    .copy()
)

train_yarn, test_yarn = train_test_split(
    yarn_count_model_data,
    test_size=0.20,
    random_state=42,
    stratify=yarn_count_model_data[
        "yarn_count_class"
    ]
)

yarn_count_text_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

yarn_count_text_model.fit(
    train_yarn["retrieval_text"],
    train_yarn["yarn_count_class"]
)

yarn_predictions = (
    yarn_count_text_model.predict(
        test_yarn["retrieval_text"]
    )
)

yarn_probabilities = (
    yarn_count_text_model.predict_proba(
        test_yarn["retrieval_text"]
    )
)

yarn_classes = (
    yarn_count_text_model
    .named_steps["model"]
    .classes_
)

print(
    "Top-1 yarn-count accuracy:",
    f"{accuracy_score(test_yarn['yarn_count_class'], yarn_predictions):.2%}"
)

top3_yarn_accuracy = top_k_accuracy_score(
    test_yarn["yarn_count_class"],
    yarn_probabilities,
    k=min(3, len(yarn_classes)),
    labels=yarn_classes
)

print(
    "Top-3 yarn-count accuracy:",
    f"{top3_yarn_accuracy:.2%}"
)

print(
    "Materials used for evaluation:",
    len(yarn_count_model_data)
)

print(
    "Materials excluded due to rare class:",
    len(yarn_count_prediction_data)
    - len(yarn_count_model_data)
)

# Retrain the yarn-count classifier using all available labeled data
final_yarn_count_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=2
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])

final_yarn_count_model.fit(
    yarn_count_model_data["retrieval_text"],
    yarn_count_model_data["yarn_count_class"]
)


In [ ]:
# Predict Top-3 yarn counts for no-attribute knitted fabrics
sap_no_attr_knit = (
    sap_no_attr_dev[
        sap_no_attr_dev["material_group_code"] == "1020001"
    ]
    .copy()
)

yarn_probabilities = (
    final_yarn_count_model.predict_proba(
        sap_no_attr_knit["retrieval_text"]
    )
)

yarn_classes = (
    final_yarn_count_model
    .named_steps["model"]
    .classes_
)

top_indices = np.argsort(
    yarn_probabilities,
    axis=1
)[:, -3:][:, ::-1]

predicted_yarn_records = []

for row_idx, material_id in enumerate(
    sap_no_attr_knit["material_id"]
):
    for rank, class_idx in enumerate(
        top_indices[row_idx],
        start=1
    ):
        predicted_yarn_records.append({
            "material_id": material_id,
            "predicted_yarn_count":
                yarn_classes[class_idx],
            "yarn_rank": rank,
            "yarn_probability":
                yarn_probabilities[row_idx, class_idx]
        })

predicted_yarn_counts = pd.DataFrame(
    predicted_yarn_records
)

display(predicted_yarn_counts.head(10))

# Prepare canonical PLM knitting yarn counts
plm_yarn_count_retrieval = (
    plm_yarn_count_knit[
        ["plm_code", "plm_yarn_count"]
    ]
    .copy()
)

plm_yarn_count_retrieval["yarn_count_class"] = (
    plm_yarn_count_retrieval["plm_yarn_count"]
    .apply(canonicalize_yarn_count)
)

plm_yarn_count_retrieval = (
    plm_yarn_count_retrieval
    .dropna(subset=["yarn_count_class"])
    .drop_duplicates("plm_code")
)

# Build PLM pool indexed by structure and yarn count
plm_knit_structure_yarn = (
    plm_candidates_enriched[
        [
            "plm_code",
            "material_group_code",
            "structured_text"
        ]
    ]
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ],
        on="plm_code",
        how="inner"
    )
    .merge(
        plm_yarn_count_retrieval[
            ["plm_code", "yarn_count_class"]
        ],
        on="plm_code",
        how="inner"
    )
)

plm_knit_structure_yarn = (
    plm_knit_structure_yarn[
        plm_knit_structure_yarn[
            "material_group_code"
        ] == "1020001"
    ]
    .copy()
)

# Combine predicted structures and yarn counts
orme_structure_predictions = (
    predicted_structures[
        predicted_structures["material_group_code"]
        == "1020001"
    ][
        [
            "material_id",
            "predicted_structure",
            "structure_rank",
            "structure_probability"
        ]
    ]
)

prediction_combinations = (
    orme_structure_predictions
    .merge(
        predicted_yarn_counts,
        on="material_id",
        how="inner"
    )
)

print(
    "Prediction combinations:",
    len(prediction_combinations)
)


In [ ]:
def generate_structure_yarn_candidates(
    sap_df,
    prediction_df,
    plm_df,
    top_k_per_combination=20
):
    records = []

    sap_lookup = (
        sap_df
        .set_index("material_id")["retrieval_text"]
        .to_dict()
    )

    for (
        structure_value,
        yarn_value
    ), predictions in prediction_df.groupby(
        [
            "predicted_structure",
            "predicted_yarn_count"
        ]
    ):

        plm_pool = (
            plm_df[
                (plm_df["structure"] == structure_value)
                & (
                    plm_df["yarn_count_class"]
                    == yarn_value
                )
            ]
            .reset_index(drop=True)
        )

        if plm_pool.empty:
            continue

        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True,
            norm="l2"
        )

        plm_matrix = vectorizer.fit_transform(
            plm_pool["structured_text"]
        )

        n_neighbors = min(
            top_k_per_combination,
            len(plm_pool)
        )

        nn_model = NearestNeighbors(
            n_neighbors=n_neighbors,
            metric="cosine",
            algorithm="brute"
        )

        nn_model.fit(plm_matrix)

        material_ids = [
            material_id
            for material_id in predictions["material_id"]
            if material_id in sap_lookup
        ]

        if not material_ids:
            continue

        sap_texts = [
            sap_lookup[material_id]
            for material_id in material_ids
        ]

        sap_matrix = vectorizer.transform(
            sap_texts
        )

        distances, indices = nn_model.kneighbors(
            sap_matrix
        )

        for row_idx, material_id in enumerate(
            material_ids
        ):
            prediction_row = (
                predictions[
                    predictions["material_id"]
                    == material_id
                ]
                .iloc[0]
            )

            for retrieval_rank, (
                plm_idx,
                distance
            ) in enumerate(
                zip(
                    indices[row_idx],
                    distances[row_idx]
                ),
                start=1
            ):
                records.append({
                    "material_id": material_id,
                    "candidate_plm_code":
                        plm_pool.loc[
                            plm_idx,
                            "plm_code"
                        ],
                    "predicted_structure":
                        structure_value,
                    "predicted_yarn_count":
                        yarn_value,
                    "structure_rank":
                        prediction_row[
                            "structure_rank"
                        ],
                    "yarn_rank":
                        prediction_row[
                            "yarn_rank"
                        ],
                    "retrieval_rank":
                        retrieval_rank,
                    "retrieval_similarity":
                        1 - distance
                })

    return pd.DataFrame(records)


In [ ]:
pseudo_structure_yarn_candidates = (
    generate_structure_yarn_candidates(
        sap_no_attr_knit,
        prediction_combinations,
        plm_knit_structure_yarn,
        top_k_per_combination=20
    )
)

pseudo_structure_yarn_pairs = (
    pseudo_structure_yarn_candidates[
        [
            "material_id",
            "candidate_plm_code"
        ]
    ]
    .drop_duplicates()
)

print(
    "Structure + yarn candidate pairs:",
    len(pseudo_structure_yarn_pairs)
)

print(
    "Average candidates per covered SAP:",
    len(pseudo_structure_yarn_pairs)
    / pseudo_structure_yarn_pairs[
        "material_id"
    ].nunique()
)

orme_dev_truth = (
    no_attr_dev_truth[
        no_attr_dev_truth["material_id"]
        .isin(set(sap_no_attr_knit["material_id"]))
    ]
)

orme_structure_yarn_recall = (
    orme_dev_truth
    .merge(
        pseudo_structure_yarn_pairs,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

print(
    "Structure + yarn ORME recall:",
    f"{orme_structure_yarn_recall['candidate_plm_code'].notna().mean():.2%}"
)

# Add structure+yarn channel to the existing no-attribute candidate pool
no_attr_candidates_v2 = (
    pd.concat(
        [
            no_attr_combined_candidates,
            pseudo_structure_yarn_pairs
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

no_attr_recall_v2 = (
    no_attr_dev_truth
    .merge(
        no_attr_candidates_v2,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

no_attr_recall_v2["retrieved"] = (
    no_attr_recall_v2[
        "candidate_plm_code"
    ].notna()
)

print(
    "Overall no-attribute recall v2:",
    f"{no_attr_recall_v2['retrieved'].mean():.2%}"
)

display(
    no_attr_recall_v2
    .merge(
        no_attribute_materials[
            [
                "material_id",
                "material_group_code"
            ]
        ],
        on="material_id",
        how="left"
    )
    .groupby("material_group_code")
    .agg(
        known_materials=("material_id", "size"),
        candidate_recall=("retrieved", "mean")
    )
)


In [ ]:
# Audit yarn-count availability on true PLM codes
orme_truth_yarn_audit = (
    orme_dev_truth
    .merge(
        plm_yarn_count_retrieval[
            [
                "plm_code",
                "yarn_count_class"
            ]
        ],
        on="plm_code",
        how="left"
    )
)

print(
    "ORME development materials:",
    len(orme_truth_yarn_audit)
)

print(
    "True PLM with yarn count:",
    f"{orme_truth_yarn_audit['yarn_count_class'].notna().mean():.2%}"
)

# Check whether predicted Top-3 yarn counts contain the true PLM yarn count
predicted_yarn_sets = (
    predicted_yarn_counts
    .groupby("material_id")[
        "predicted_yarn_count"
    ]
    .apply(set)
    .rename("predicted_yarn_set")
    .reset_index()
)

orme_yarn_prediction_audit = (
    orme_truth_yarn_audit
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
)

orme_yarn_prediction_audit[
    "true_yarn_in_top3"
] = (
    orme_yarn_prediction_audit.apply(
        lambda row:
            (
                row["yarn_count_class"]
                in row["predicted_yarn_set"]
            )
            if (
                pd.notna(row["yarn_count_class"])
                and isinstance(
                    row["predicted_yarn_set"],
                    set
                )
            )
            else np.nan,
        axis=1
    )
)

valid_yarn_audit = (
    orme_yarn_prediction_audit[
        orme_yarn_prediction_audit[
            "true_yarn_in_top3"
        ].notna()
    ]
)

print(
    "Comparable materials:",
    len(valid_yarn_audit)
)

print(
    "Predicted yarn Top-3 agreement with true PLM:",
    f"{valid_yarn_audit['true_yarn_in_top3'].mean():.2%}"
)


In [ ]:
# Get true PLM fabric structure
orme_truth_structure_audit = (
    orme_dev_truth
    .merge(
        plm_structure_candidates[
            [
                "plm_code",
                "structure"
            ]
        ].rename(
            columns={
                "structure":
                "true_plm_structure"
            }
        ),
        on="plm_code",
        how="left"
    )
)

predicted_structure_sets = (
    predicted_structures[
        predicted_structures[
            "material_group_code"
        ] == "1020001"
    ]
    .groupby("material_id")[
        "predicted_structure"
    ]
    .apply(set)
    .rename("predicted_structure_set")
    .reset_index()
)

orme_structure_prediction_audit = (
    orme_truth_structure_audit
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
)

orme_structure_prediction_audit[
    "true_structure_in_top3"
] = (
    orme_structure_prediction_audit.apply(
        lambda row:
            (
                row["true_plm_structure"]
                in row["predicted_structure_set"]
            )
            if (
                pd.notna(row["true_plm_structure"])
                and isinstance(
                    row["predicted_structure_set"],
                    set
                )
            )
            else np.nan,
        axis=1
    )
)

valid_structure_audit = (
    orme_structure_prediction_audit[
        orme_structure_prediction_audit[
            "true_structure_in_top3"
        ].notna()
    ]
)

print(
    "Predicted structure Top-3 agreement with true PLM:",
    f"{valid_structure_audit['true_structure_in_top3'].mean():.2%}"
)


In [ ]:
# Audit whether both true PLM structure and yarn count
# are contained in the predicted Top-3 sets
orme_joint_audit = (
    orme_truth_yarn_audit[
        [
            "material_id",
            "plm_code",
            "yarn_count_class"
        ]
    ]
    .merge(
        orme_truth_structure_audit[
            [
                "material_id",
                "true_plm_structure"
            ]
        ],
        on="material_id",
        how="left"
    )
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
)

orme_joint_audit["true_yarn_in_top3"] = (
    orme_joint_audit.apply(
        lambda row:
            row["yarn_count_class"] in row["predicted_yarn_set"]
            if (
                pd.notna(row["yarn_count_class"])
                and isinstance(row["predicted_yarn_set"], set)
            )
            else np.nan,
        axis=1
    )
)

orme_joint_audit["true_structure_in_top3"] = (
    orme_joint_audit.apply(
        lambda row:
            row["true_plm_structure"] in row["predicted_structure_set"]
            if (
                pd.notna(row["true_plm_structure"])
                and isinstance(row["predicted_structure_set"], set)
            )
            else np.nan,
        axis=1
    )
)

comparable_joint = orme_joint_audit[
    orme_joint_audit["true_yarn_in_top3"].notna()
    & orme_joint_audit["true_structure_in_top3"].notna()
].copy()

comparable_joint["both_in_top3"] = (
    comparable_joint["true_yarn_in_top3"]
    & comparable_joint["true_structure_in_top3"]
)

print(
    "Comparable ORME materials:",
    len(comparable_joint)
)

print(
    "Both true structure and yarn in predicted Top-3:",
    f"{comparable_joint['both_in_top3'].mean():.2%}"
)


In [ ]:
# Test candidate-pool recall before text ranking
true_plm_attributes = (
    orme_dev_truth
    .merge(
        plm_structure_candidates[
            ["plm_code", "structure"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        plm_yarn_count_retrieval[
            ["plm_code", "yarn_count_class"]
        ],
        on="plm_code",
        how="left"
    )
    .merge(
        predicted_structure_sets,
        on="material_id",
        how="left"
    )
    .merge(
        predicted_yarn_sets,
        on="material_id",
        how="left"
    )
)

true_plm_attributes["eligible_by_pseudo_attributes"] = (
    true_plm_attributes.apply(
        lambda row:
            (
                row["structure"]
                in row["predicted_structure_set"]
            )
            and (
                row["yarn_count_class"]
                in row["predicted_yarn_set"]
            )
            if (
                pd.notna(row["structure"])
                and pd.notna(row["yarn_count_class"])
                and isinstance(
                    row["predicted_structure_set"],
                    set
                )
                and isinstance(
                    row["predicted_yarn_set"],
                    set
                )
            )
            else False,
        axis=1
    )
)

print(
    "Pseudo-attribute pool recall:",
    f"{true_plm_attributes['eligible_by_pseudo_attributes'].mean():.2%}"
)

# Build the full pseudo-attribute candidate pool
pseudo_attribute_candidates = (
    prediction_combinations[
        [
            "material_id",
            "predicted_structure",
            "structure_rank",
            "structure_probability",
            "predicted_yarn_count",
            "yarn_rank",
            "yarn_probability"
        ]
    ]
    .merge(
        plm_knit_structure_yarn[
            [
                "plm_code",
                "structure",
                "yarn_count_class"
            ]
        ],
        left_on=[
            "predicted_structure",
            "predicted_yarn_count"
        ],
        right_on=[
            "structure",
            "yarn_count_class"
        ],
        how="inner"
    )
    .rename(
        columns={
            "plm_code": "candidate_plm_code"
        }
    )
)


In [ ]:
# Keep one row per SAP-PLM pair
pseudo_attribute_candidates_clean = (
    pseudo_attribute_candidates
    .groupby(
        [
            "material_id",
            "candidate_plm_code"
        ],
        as_index=False
    )
    .agg(
        structure_probability=(
            "structure_probability",
            "max"
        ),
        yarn_probability=(
            "yarn_probability",
            "max"
        ),
        best_structure_rank=(
            "structure_rank",
            "min"
        ),
        best_yarn_rank=(
            "yarn_rank",
            "min"
        )
    )
)

pseudo_attribute_candidates_clean[
    "pseudo_attribute_score"
] = (
    pseudo_attribute_candidates_clean[
        "structure_probability"
    ]
    * pseudo_attribute_candidates_clean[
        "yarn_probability"
    ]
)

print(
    "Pseudo-attribute candidate pairs:",
    len(pseudo_attribute_candidates_clean)
)

print(
    "Covered SAP materials:",
    pseudo_attribute_candidates_clean[
        "material_id"
    ].nunique()
)

print(
    "Average candidates per covered SAP:",
    len(pseudo_attribute_candidates_clean)
    / pseudo_attribute_candidates_clean[
        "material_id"
    ].nunique()
)

# Combine text retrieval and pseudo-attribute candidate generation
no_attr_knit_candidate_union = (
    pd.concat(
        [
            multi_channel_union[
                multi_channel_union["material_id"]
                .isin(set(sap_no_attr_knit["material_id"]))
            ],
            pseudo_attribute_candidates_clean[
                [
                    "material_id",
                    "candidate_plm_code"
                ]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

orme_union_recall = (
    orme_dev_truth
    .merge(
        no_attr_knit_candidate_union,
        left_on=[
            "material_id",
            "plm_code"
        ],
        right_on=[
            "material_id",
            "candidate_plm_code"
        ],
        how="left"
    )
)

orme_union_recall["retrieved"] = (
    orme_union_recall[
        "candidate_plm_code"
    ].notna()
)

print(
    "ORME candidate union recall:",
    f"{orme_union_recall['retrieved'].mean():.2%}"
)

print(
    "Average union candidates per SAP:",
    len(no_attr_knit_candidate_union)
    / no_attr_knit_candidate_union[
        "material_id"
    ].nunique()
)


In [ ]:
# Final no-attribute candidate union for development
no_attr_candidate_union_v3 = (
    pd.concat(
        [
            no_attr_combined_candidates,
            pseudo_attribute_candidates_clean[
                ["material_id", "candidate_plm_code"]
            ]
        ],
        ignore_index=True
    )
    .drop_duplicates()
)

print(
    "Final no-attribute candidate pairs:",
    len(no_attr_candidate_union_v3)
)

print(
    "Average candidates per SAP:",
    len(no_attr_candidate_union_v3)
    / no_attr_candidate_union_v3["material_id"].nunique()
)

# Aggregate text retrieval features
text_pair_features = (
    multi_channel_candidates
    .groupby(
        [
            "material_id",
            "candidate_plm_code",
            "channel"
        ],
        as_index=False
    )
    .agg(
        similarity=("channel_similarity", "max"),
        rank=("channel_rank", "min")
    )
)

text_similarity_wide = (
    text_pair_features
    .pivot(
        index=["material_id", "candidate_plm_code"],
        columns="channel",
        values="similarity"
    )
    .add_suffix("_similarity")
    .reset_index()
)

text_rank_wide = (
    text_pair_features
    .pivot(
        index=["material_id", "candidate_plm_code"],
        columns="channel",
        values="rank"
    )
    .add_suffix("_rank")
    .reset_index()
)

text_channel_count = (
    text_pair_features
    .groupby(
        ["material_id", "candidate_plm_code"]
    )["channel"]
    .nunique()
    .rename("text_channel_count")
    .reset_index()
)

# Aggregate pseudo-structure retrieval features
pseudo_structure_features = (
    pseudo_structure_candidates
    .groupby(
        ["material_id", "candidate_plm_code"],
        as_index=False
    )
    .agg(
        structure_probability=(
            "structure_probability",
            "max"
        ),
        best_structure_rank=(
            "structure_rank",
            "min"
        ),
        structure_retrieval_similarity=(
            "structure_retrieval_similarity",
            "max"
        ),
        structure_retrieval_rank=(
            "structure_retrieval_rank",
            "min"
        )
    )
)

pseudo_structure_features[
    "from_pseudo_structure"
] = 1

pseudo_attribute_features = (
    pseudo_attribute_candidates_clean.copy()
)

pseudo_attribute_features[
    "from_structure_yarn"
] = 1


In [ ]:
# Build one-row-per-SAP-PLM candidate master
no_attr_candidate_master = (
    no_attr_candidate_union_v3
    .merge(
        text_similarity_wide,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        text_rank_wide,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        text_channel_count,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        pseudo_structure_features,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        pseudo_attribute_features,
        on=["material_id", "candidate_plm_code"],
        how="left",
        validate="one_to_one"
    )
)

for column in [
    "text_channel_count",
    "from_pseudo_structure",
    "from_structure_yarn"
]:
    no_attr_candidate_master[column] = (
        no_attr_candidate_master[column]
        .fillna(0)
    )

assert not no_attr_candidate_master.duplicated(
    ["material_id", "candidate_plm_code"]
).any()

# Add material group
no_attr_candidate_master = (
    no_attr_candidate_master
    .merge(
        material_group_lookup,
        on="material_id",
        how="left",
        validate="many_to_one"
    )
)

# Add known target PLM for development materials only
no_attr_dev_truth_labeled = (
    no_attr_dev_truth
    .rename(
        columns={
            "plm_code": "true_plm_code"
        }
    )
)

no_attr_dev_pairs = (
    no_attr_candidate_master[
        no_attr_candidate_master["material_id"]
        .isin(no_attr_dev_ids)
    ]
    .merge(
        no_attr_dev_truth_labeled,
        on="material_id",
        how="inner",
        validate="many_to_one"
    )
)

no_attr_dev_pairs["is_match"] = (
    no_attr_dev_pairs["candidate_plm_code"]
    == no_attr_dev_pairs["true_plm_code"]
).astype(int)

print(
    "Development SAP materials:",
    no_attr_dev_pairs["material_id"].nunique()
)

print(
    "Candidate pairs:",
    len(no_attr_dev_pairs)
)

print(
    "Positive pairs retrieved:",
    no_attr_dev_pairs["is_match"].sum()
)

candidate_recall = (
    no_attr_dev_pairs
    .groupby("material_id")["is_match"]
    .max()
    .mean()
)

print(
    "Candidate recall:",
    f"{candidate_recall:.2%}"
)


In [ ]:
candidate_feature_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if (
        column.endswith("_similarity")
        or column.endswith("_rank")
        or column in [
            "text_channel_count",
            "structure_probability",
            "yarn_probability",
            "pseudo_attribute_score",
            "from_pseudo_structure",
            "from_structure_yarn"
        ]
    )
]

print(candidate_feature_columns)

# Inspect duplicated pseudo-structure feature names
print([
    column
    for column in no_attr_dev_pairs.columns
    if "structure_probability" in column
    or "best_structure_rank" in column
])

# Consolidate structure probability from multiple candidate channels
structure_probability_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if column.startswith("structure_probability")
]

structure_rank_columns = [
    column
    for column in no_attr_dev_pairs.columns
    if column.startswith("best_structure_rank")
]

no_attr_dev_pairs["structure_probability"] = (
    no_attr_dev_pairs[
        structure_probability_columns
    ]
    .max(axis=1)
)

no_attr_dev_pairs["best_structure_rank"] = (
    no_attr_dev_pairs[
        structure_rank_columns
    ]
    .min(axis=1)
)

# Add material-group indicators
for group_code in [
    "1020001",
    "1020002",
    "1030004"
]:
    no_attr_dev_pairs[
        f"group_{group_code}"
    ] = (
        no_attr_dev_pairs[
            "material_group_code"
        ] == group_code
    ).astype(int)

no_attr_features = [
    "structured_text_similarity",
    "enriched_text_similarity",
    "multi_value_text_similarity",
    "original_text_similarity",

    "structure_retrieval_similarity",
    "structure_probability",
    "best_structure_rank",

    "yarn_probability",
    "best_yarn_rank",
    "pseudo_attribute_score",

    "text_channel_count",
    "from_pseudo_structure",
    "from_structure_yarn",

    "group_1020001",
    "group_1020002",
    "group_1030004"
]

# Create material-level development split
no_attr_dev_material_table = (
    no_attr_development[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .copy()
)

no_attr_train_materials, no_attr_validation_materials = (
    train_test_split(
        no_attr_dev_material_table,
        test_size=0.20,
        random_state=777,
        stratify=no_attr_dev_material_table[
            "material_group_code"
        ]
    )
)

no_attr_train_ids = set(
    no_attr_train_materials["material_id"]
)

no_attr_validation_ids = set(
    no_attr_validation_materials["material_id"]
)

print(
    "Train materials:",
    len(no_attr_train_ids)
)

print(
    "Validation materials:",
    len(no_attr_validation_ids)
)


In [ ]:
# Identify train materials with a retrieved positive candidate
train_pair_pool = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_train_ids)
    ]
    .copy()
)

retrieved_train_ids = set(
    train_pair_pool
    .groupby("material_id")["is_match"]
    .max()
    .loc[lambda x: x == 1]
    .index
)

train_pair_pool = (
    train_pair_pool[
        train_pair_pool["material_id"]
        .isin(retrieved_train_ids)
    ]
    .copy()
)

print(
    "Train materials with true PLM retrieved:",
    len(retrieved_train_ids)
)

# Score used only for hard-negative selection
train_pair_pool["hardness_score"] = (
    train_pair_pool[
        "structured_text_similarity"
    ].fillna(0)

    + train_pair_pool[
        "enriched_text_similarity"
    ].fillna(0)

    + train_pair_pool[
        "structure_retrieval_similarity"
    ].fillna(0)

    + train_pair_pool[
        "pseudo_attribute_score"
    ].fillna(0)

    + 0.25 * train_pair_pool[
        "multi_value_text_similarity"
    ].fillna(0)
)

positive_pairs = (
    train_pair_pool[
        train_pair_pool["is_match"] == 1
    ]
    .copy()
)

negative_pairs = (
    train_pair_pool[
        train_pair_pool["is_match"] == 0
    ]
    .copy()
)

hard_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "hardness_score"],
        ascending=[True, False]
    )
    .groupby("material_id")
    .head(30)
)

rng = np.random.default_rng(42)

negative_pairs["random_score"] = (
    rng.random(len(negative_pairs))
)

random_negatives = (
    negative_pairs
    .sort_values(
        ["material_id", "random_score"]
    )
    .groupby("material_id")
    .head(20)
)

no_attr_training_sample = (
    pd.concat(
        [
            positive_pairs,
            hard_negatives,
            random_negatives
        ],
        ignore_index=True
    )
    .drop_duplicates(
        ["material_id", "candidate_plm_code"]
    )
)

print(
    "Training rows:",
    len(no_attr_training_sample)
)

print(
    "Training positives:",
    no_attr_training_sample[
        "is_match"
    ].sum()
)


In [ ]:
# Train first no-attribute reranker
no_attr_rf_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=-1,
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        )
    )
])

no_attr_rf_model.fit(
    no_attr_training_sample[
        no_attr_features
    ],
    no_attr_training_sample[
        "is_match"
    ]
)

# Score the full validation candidate pool
no_attr_validation_pairs = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_validation_ids)
    ]
    .copy()
)

no_attr_validation_pairs["score"] = (
    no_attr_rf_model.predict_proba(
        no_attr_validation_pairs[
            no_attr_features
        ]
    )[:, 1]
)

no_attr_validation_pairs["rank"] = (
    no_attr_validation_pairs
    .groupby("material_id")["score"]
    .rank(
        ascending=False,
        method="first"
    )
)

validation_true_ranks = (
    no_attr_validation_pairs[
        no_attr_validation_pairs["is_match"] == 1
    ][
        [
            "material_id",
            "material_group_code",
            "rank"
        ]
    ]
    .copy()
)

print(
    "Conditional Top-1:",
    f"{(validation_true_ranks['rank'] <= 1).mean():.2%}"
)

print(
    "Conditional Top-3:",
    f"{(validation_true_ranks['rank'] <= 3).mean():.2%}"
)

print(
    "Conditional Top-5:",
    f"{(validation_true_ranks['rank'] <= 5).mean():.2%}"
)

print(
    "Conditional Top-10:",
    f"{(validation_true_ranks['rank'] <= 10).mean():.2%}"
)

print(
    "Conditional MRR:",
    f"{(1 / validation_true_ranks['rank']).mean():.4f}"
)


In [ ]:
# End-to-end validation evaluation
validation_rank_table = (
    no_attr_validation_materials[
        [
            "material_id",
            "material_group_code"
        ]
    ]
    .merge(
        validation_true_ranks[
            [
                "material_id",
                "rank"
            ]
        ],
        on="material_id",
        how="left"
    )
)

print(
    "Validation candidate recall:",
    f"{validation_rank_table['rank'].notna().mean():.2%}"
)

for k in [1, 3, 5, 10]:
    success = (
        validation_rank_table["rank"]
        .le(k)
        .fillna(False)
        .mean()
    )

    print(
        f"End-to-end Top-{k}:",
        f"{success:.2%}"
    )

# Evaluate no-attribute reranker by material group
group_results = []

for group_code, group in validation_rank_table.groupby(
    "material_group_code"
):
    retrieved = group[group["rank"].notna()]

    group_results.append({
        "material_group_code": group_code,
        "materials": len(group),

        "candidate_recall":
            group["rank"].notna().mean(),

        "conditional_top1":
            (retrieved["rank"] <= 1).mean(),

        "conditional_top3":
            (retrieved["rank"] <= 3).mean(),

        "conditional_top5":
            (retrieved["rank"] <= 5).mean(),

        "end_to_end_top5":
            group["rank"]
            .le(5)
            .fillna(False)
            .mean()
    })

display(pd.DataFrame(group_results))

# Inspect feature importance
imputer = no_attr_rf_model.named_steps["imputer"]
rf_model = no_attr_rf_model.named_steps["model"]

feature_names = imputer.get_feature_names_out(
    no_attr_features
)

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf_model.feature_importances_
    })
    .sort_values(
        "importance",
        ascending=False
    )
)

display(feature_importance.head(25))

# Build a canonical fiber lexicon
fiber_alias_mapping = {
    "PAMUK": "COTTON",
    "COTTON": "COTTON",

    "PES": "POLYESTER",
    "POLYESTER": "POLYESTER",
    "POLIESTER": "POLYESTER",

    "VIS": "VISCOSE",
    "VISCOSE": "VISCOSE",

    "EA": "ELASTANE",
    "ELASTAN": "ELASTANE",
    "ELASTANE": "ELASTANE",
    "ELASTHANE": "ELASTANE",
    "SPANDEX": "ELASTANE",

    "PA": "POLYAMIDE",
    "POLYAMIDE": "POLYAMIDE",
    "POLYAMIDE6": "POLYAMIDE",

    "LYOCELL": "LYOCELL",
    "TENCEL": "LYOCELL",

    "ACRYLIC": "ACRYLIC",
    "AKRILIK": "ACRYLIC",

    "LINEN": "LINEN",
    "KETEN": "LINEN",

    "WOOL": "WOOL",
    "YUN": "WOOL",
    "YÜN": "WOOL"
}


In [ ]:
def extract_fiber_names_from_text(text):
    if pd.isna(text):
        return set()

    text = str(text).upper()

    found = set()

    for alias, canonical in fiber_alias_mapping.items():

        # Short abbreviations need token boundaries
        pattern = rf"(?<![A-ZÇĞİÖŞÜ]){re.escape(alias)}(?![A-ZÇĞİÖŞÜ])"

        if re.search(pattern, text):
            found.add(canonical)

    return found

# Compare description-derived fibers with actual SAP fiber attributes
fiber_extraction_audit = (
    sap_candidates_source[
        [
            "material_id",
            "material_group_code",
            "retrieval_text"
        ]
    ]
    .merge(
        sap_fiber_sets,
        on="material_id",
        how="inner"
    )
)

fiber_extraction_audit[
    "predicted_fiber_set"
] = (
    fiber_extraction_audit[
        "retrieval_text"
    ]
    .apply(extract_fiber_names_from_text)
)

fiber_extraction_audit[
    "actual_fiber_set"
] = (
    fiber_extraction_audit[
        "fiber_key"
    ]
    .apply(set)
)

fiber_extraction_audit[
    "fiber_set_match"
] = (
    fiber_extraction_audit[
        "predicted_fiber_set"
    ]
    == fiber_extraction_audit[
        "actual_fiber_set"
    ]
)

fiber_extraction_audit[
    "fiber_jaccard"
] = [
    (
        len(pred & actual)
        / len(pred | actual)
        if pred and actual
        else np.nan
    )
    for pred, actual in zip(
        fiber_extraction_audit[
            "predicted_fiber_set"
        ],
        fiber_extraction_audit[
            "actual_fiber_set"
        ]
    )
]

print(
    "Materials:",
    len(fiber_extraction_audit)
)

print(
    "Description with at least one extracted fiber:",
    f"{fiber_extraction_audit['predicted_fiber_set'].apply(bool).mean():.2%}"
)

print(
    "Exact fiber-set match:",
    f"{fiber_extraction_audit['fiber_set_match'].mean():.2%}"
)

print(
    "Mean fiber Jaccard:",
    f"{fiber_extraction_audit['fiber_jaccard'].mean():.3f}"
)

display(
    fiber_extraction_audit
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        extraction_coverage=(
            "predicted_fiber_set",
            lambda x: x.apply(bool).mean()
        ),
        exact_match=(
            "fiber_set_match",
            "mean"
        ),
        mean_jaccard=(
            "fiber_jaccard",
            "mean"
        )
    )
)


In [ ]:
# Inspect validation candidate-pool size by material group
validation_candidate_stats = (
    no_attr_dev_pairs[
        no_attr_dev_pairs["material_id"]
        .isin(no_attr_validation_ids)
    ]
    .groupby(
        ["material_id", "material_group_code"],
        as_index=False
    )
    .agg(
        candidate_count=("candidate_plm_code", "size"),
        true_retrieved=("is_match", "max")
    )
)

candidate_pool_summary = (
    validation_candidate_stats
    .groupby("material_group_code")
    .agg(
        materials=("material_id", "size"),
        candidate_recall=("true_retrieved", "mean"),
        mean_candidates=("candidate_count", "mean"),
        median_candidates=("candidate_count", "median"),
        p90_candidates=(
            "candidate_count",
            lambda x: x.quantile(0.90)
        )
    )
)

display(candidate_pool_summary)

group_specific_features = [
    feature
    for feature in no_attr_features
    if not feature.startswith("group_")
]

print(group_specific_features)

def build_no_attr_lr():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ])


def build_no_attr_rf():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
                add_indicator=True
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=42
            )
        )
    ])


In [ ]:
def evaluate_group_reranker(
    group_code,
    model,
    model_name,
    features
):
    train_group = (
        no_attr_training_sample[
            (
                no_attr_training_sample["material_id"]
                .isin(no_attr_train_ids)
            )
            & (
                no_attr_training_sample[
                    "material_group_code"
                ] == group_code
            )
        ]
        .copy()
    )

    validation_group = (
        no_attr_dev_pairs[
            (
                no_attr_dev_pairs["material_id"]
                .isin(no_attr_validation_ids)
            )
            & (
                no_attr_dev_pairs[
                    "material_group_code"
                ] == group_code
            )
        ]
        .copy()
    )

    validation_materials_group = (
        no_attr_validation_materials[
            no_attr_validation_materials[
                "material_group_code"
            ] == group_code
        ][["material_id"]]
        .copy()
    )

    model.fit(
        train_group[features],
        train_group["is_match"]
    )

    validation_group["score"] = (
        model.predict_proba(
            validation_group[features]
        )[:, 1]
    )

    validation_group["rank"] = (
        validation_group
        .groupby("material_id")["score"]
        .rank(
            ascending=False,
            method="first"
        )
    )

    true_ranks = (
        validation_group[
            validation_group["is_match"] == 1
        ][
            ["material_id", "rank"]
        ]
        .copy()
    )

    evaluation_table = (
        validation_materials_group
        .merge(
            true_ranks,
            on="material_id",
            how="left",
            validate="one_to_one"
        )
    )

    retrieved = (
        evaluation_table[
            evaluation_table["rank"].notna()
        ]
    )

    result = {
        "material_group_code": group_code,
        "model": model_name,

        "validation_materials":
            len(evaluation_table),

        "candidate_recall":
            evaluation_table["rank"]
            .notna()
            .mean(),

        "conditional_top1":
            (retrieved["rank"] <= 1).mean(),

        "conditional_top3":
            (retrieved["rank"] <= 3).mean(),

        "conditional_top5":
            (retrieved["rank"] <= 5).mean(),

        "conditional_top10":
            (retrieved["rank"] <= 10).mean(),

        "conditional_mrr":
            (1 / retrieved["rank"]).mean(),

        "end_to_end_top1":
            evaluation_table["rank"]
            .le(1)
            .fillna(False)
            .mean(),

        "end_to_end_top3":
            evaluation_table["rank"]
            .le(3)
            .fillna(False)
            .mean(),

        "end_to_end_top5":
            evaluation_table["rank"]
            .le(5)
            .fillna(False)
            .mean(),

        "end_to_end_top10":
            evaluation_table["rank"]
            .le(10)
            .fillna(False)
            .mean()
    }

    return result, model, validation_group


In [ ]:
group_model_results = []

trained_group_models = {}
scored_group_validation = {}

group_codes = [
    "1020001",  # ORME
    "1020002"   # DOKUMA
]

for group_code in group_codes:

    lr_result, lr_model, lr_scored = (
        evaluate_group_reranker(
            group_code=group_code,
            model=build_no_attr_lr(),
            model_name="Logistic Regression",
            features=group_specific_features
        )
    )

    group_model_results.append(lr_result)

    trained_group_models[
        (group_code, "LR")
    ] = lr_model

    scored_group_validation[
        (group_code, "LR")
    ] = lr_scored


    rf_result, rf_model, rf_scored = (
        evaluate_group_reranker(
            group_code=group_code,
            model=build_no_attr_rf(),
            model_name="Random Forest",
            features=group_specific_features
        )
    )

    group_model_results.append(rf_result)

    trained_group_models[
        (group_code, "RF")
    ] = rf_model

    scored_group_validation[
        (group_code, "RF")
    ] = rf_scored

group_model_results_df = (
    pd.DataFrame(group_model_results)
    .sort_values(
        [
            "material_group_code",
            "conditional_mrr"
        ],
        ascending=[True, False]
    )
)

display(group_model_results_df)

rank_distribution_results = []

for (
    group_code,
    model_name
), scored_data in scored_group_validation.items():

    true_ranks = (
        scored_data[
            scored_data["is_match"] == 1
        ]["rank"]
    )

    rank_distribution_results.append({
        "material_group_code":
            group_code,

        "model":
            model_name,

        "median_rank":
            true_ranks.median(),

        "p75_rank":
            true_ranks.quantile(0.75),

        "p90_rank":
            true_ranks.quantile(0.90),

        "mean_rank":
            true_ranks.mean(),

        "max_rank":
            true_ranks.max()
    })

display(
    pd.DataFrame(
        rank_distribution_results
    )
)


## 8. PLM exact-profile duplicate detection and cleanup prioritization

This stage is independent of SAP→PLM matching. It builds a technical signature from PLM single-value and multi-value characteristics, detects repeated exact observed profiles, and then prioritizes review using status, technical completeness, and SAP usage.

A repeated profile is **not automatically treated as a confirmed business duplicate**. Low-information profiles can collide simply because too few technical fields are populated.


In [ ]:
# Audit PLM master columns before duplicate detection
plm_column_audit = pd.DataFrame({
    "column": plm_codes.columns,
    "dtype": [
        str(plm_codes[column].dtype)
        for column in plm_codes.columns
    ],
    "non_null": [
        plm_codes[column].notna().sum()
        for column in plm_codes.columns
    ],
    "unique_values": [
        plm_codes[column].nunique(dropna=True)
        for column in plm_codes.columns
    ]
})

display(plm_column_audit)

def normalize_duplicate_text(series):
    return (
        series
        .astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .fillna("<MISSING>")
        .replace("", "<MISSING>")
    )


def normalize_duplicate_column(series):
    # Preserve numeric equivalence such as 150 and 150.0
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(
            series,
            errors="coerce"
        )

        return (
            numeric
            .round(8)
            .astype("string")
            .fillna("<MISSING>")
        )

    return normalize_duplicate_text(series)

# Inspect PLM-code collisions introduced by normalization
plm_code_collision_audit = (
    plm_codes[
        ["PLM Kodu"]
    ]
    .assign(
        normalized_plm_code=(
            plm_codes["PLM Kodu"]
            .astype("string")
            .str.strip()
            .str.upper()
            .str.replace(r"\.0$", "", regex=True)
        )
    )
)

collision_codes = (
    plm_code_collision_audit
    .groupby("normalized_plm_code")
    .agg(
        raw_code_count=("PLM Kodu", "nunique"),
        raw_codes=(
            "PLM Kodu",
            lambda x: list(x.astype(str).unique())
        )
    )
    .query("raw_code_count > 1")
)

display(collision_codes)

# Inspect the raw PLM rows that collide after case normalization
collision_raw_codes = (
    plm_code_collision_audit[
        plm_code_collision_audit["normalized_plm_code"]
        == "DOBBY, WOVEN-OTHERS, 75 D X 75"
    ]["PLM Kodu"]
    .tolist()
)

display(
    plm_codes[
        plm_codes["PLM Kodu"].isin(collision_raw_codes)
    ].T
)

# Check whether the colliding raw identifiers exist in the multi-value file
mv_raw_codes = (
    plm_multi_value_valid["plm_code"]
    .astype("string")
    .str.strip()
)

for raw_code in collision_raw_codes:
    count = (
        mv_raw_codes
        == str(raw_code).strip()
    ).sum()

    print(
        repr(raw_code),
        "multi-value rows:",
        count
    )

def normalize_plm_code_identity(series):
    return (
        series
        .astype("string")
        .str.strip()
    )


In [ ]:
plm_duplicate_base = plm_codes.copy()

plm_duplicate_base["plm_code"] = (
    normalize_plm_code_identity(
        plm_duplicate_base["PLM Kodu"]
    )
)

print(
    "Rows:",
    len(plm_duplicate_base)
)

print(
    "Unique PLM codes:",
    plm_duplicate_base["plm_code"].nunique()
)

assert plm_duplicate_base["plm_code"].is_unique

mv_duplicate = plm_multi_value_valid.copy()

mv_duplicate["plm_code"] = (
    normalize_plm_code_identity(
        mv_duplicate["plm_code"]
    )
)

mv_duplicate["characteristic"] = (
    normalize_duplicate_text(
        mv_duplicate[
            "PLM Karakteristik Tanımı"
        ]
    )
)

mv_duplicate["value"] = (
    normalize_duplicate_text(
        mv_duplicate[
            "PLM Karakteristik Değeri"
        ]
    )
)

master_codes = set(
    plm_duplicate_base["plm_code"]
)

multi_value_codes = set(
    mv_duplicate["plm_code"]
)

print(
    "Master PLM codes:",
    len(master_codes)
)

print(
    "Multi-value PLM codes:",
    len(multi_value_codes)
)

print(
    "Multi-value codes found in master:",
    len(
        multi_value_codes
        & master_codes
    )
)

print(
    "Multi-value codes NOT found in master:",
    len(
        multi_value_codes
        - master_codes
    )
)

duplicate_excluded_columns = {
    # Identity
    "PLM Kodu",
    "plm_code",

    # Descriptive fields
    "Türkçe malzeme açıklaması",
    "Malzeme Türkçe Adı",
    "Malzeme ingilizce adı",

    # Administrative / derived fields
    "Malzeme statüs",
    "material_group_code",
    "material_group_name",
    "material_family"
}

structured_duplicate_features = [
    column
    for column in plm_duplicate_base.columns
    if column not in duplicate_excluded_columns
]

print(
    "Structured duplicate features:",
    len(structured_duplicate_features)
)

print(structured_duplicate_features)

structured_signature = (
    plm_duplicate_base[
        ["plm_code"] + structured_duplicate_features
    ]
    .copy()
)

for column in structured_duplicate_features:
    structured_signature[column] = (
        normalize_duplicate_column(
            structured_signature[column]
        )
    )

assert structured_signature["plm_code"].is_unique

print(
    "Structured PLM rows:",
    len(structured_signature)
)


In [ ]:
mv_characteristic_sets = (
    mv_duplicate
    .groupby(
        [
            "plm_code",
            "characteristic"
        ]
    )["value"]
    .agg(
        lambda values:
            " || ".join(
                sorted(set(values))
            )
    )
    .reset_index()
)

mv_signature_wide = (
    mv_characteristic_sets
    .pivot(
        index="plm_code",
        columns="characteristic",
        values="value"
    )
    .fillna("<ABSENT>")
    .reset_index()
)

mv_feature_columns = [
    column
    for column in mv_signature_wide.columns
    if column != "plm_code"
]

mv_signature_wide = (
    mv_signature_wide.rename(
        columns={
            column: f"MV__{column}"
            for column in mv_feature_columns
        }
    )
)

assert mv_signature_wide["plm_code"].is_unique

print(
    "Multi-value PLM profiles:",
    len(mv_signature_wide)
)

print(
    "Multi-value characteristics:",
    len(mv_feature_columns)
)

plm_duplicate_master = (
    structured_signature
    .merge(
        mv_signature_wide,
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

mv_columns = [
    column
    for column in plm_duplicate_master.columns
    if column.startswith("MV__")
]

plm_duplicate_master[mv_columns] = (
    plm_duplicate_master[mv_columns]
    .fillna("<ABSENT>")
)

assert plm_duplicate_master["plm_code"].is_unique

# The merge must not add or drop rows relative to the structured signature.
assert len(plm_duplicate_master) == len(structured_signature)

print(
    "Merged PLM rows:",
    len(plm_duplicate_master)
)

print(
    "Structured features:",
    len(structured_duplicate_features)
)

print(
    "Multi-value features:",
    len(mv_columns)
)

duplicate_feature_columns = (
    structured_duplicate_features
    + mv_columns
)

plm_duplicate_master[
    "material_profile_hash"
] = (
    pd.util.hash_pandas_object(
        plm_duplicate_master[
            duplicate_feature_columns
        ],
        index=False
    )
    .astype("uint64")
)


In [ ]:
profile_counts = (
    plm_duplicate_master[
        "material_profile_hash"
    ]
    .value_counts()
)

duplicate_hashes = set(
    profile_counts[
        profile_counts > 1
    ].index
)

exact_duplicate_plms = (
    plm_duplicate_master[
        plm_duplicate_master[
            "material_profile_hash"
        ].isin(duplicate_hashes)
    ]
    .copy()
)

print(
    "Total PLM codes:",
    len(plm_duplicate_master)
)

print(
    "Unique technical profiles:",
    plm_duplicate_master[
        "material_profile_hash"
    ].nunique()
)

print(
    "Exact duplicate groups:",
    len(duplicate_hashes)
)

print(
    "PLM codes inside exact duplicate groups:",
    len(exact_duplicate_plms)
)

print(
    "Duplicate PLM rate:",
    f"{len(exact_duplicate_plms) / len(plm_duplicate_master):.2%}"
)

duplicate_verification = (
    exact_duplicate_plms
    .groupby("material_profile_hash")[duplicate_feature_columns]
    .apply(
        lambda group: group.drop_duplicates().shape[0]
    )
)

print(
    "Groups with more than one distinct feature row:",
    (duplicate_verification > 1).sum()
)

assert (duplicate_verification == 1).all()

duplicate_hash_order = (
    exact_duplicate_plms[
        "material_profile_hash"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

duplicate_group_mapping = {
    profile_hash: f"PLM_DUP_{index:05d}"
    for index, profile_hash in enumerate(
        duplicate_hash_order,
        start=1
    )
}

exact_duplicate_plms[
    "duplicate_group_id"
] = (
    exact_duplicate_plms[
        "material_profile_hash"
    ]
    .map(duplicate_group_mapping)
)

exact_duplicate_plms[
    "duplicate_group_size"
] = (
    exact_duplicate_plms
    .groupby("duplicate_group_id")[
        "plm_code"
    ]
    .transform("size")
)


In [ ]:
duplicate_review = (
    exact_duplicate_plms[
        [
            "duplicate_group_id",
            "duplicate_group_size",
            "plm_code",
            "material_profile_hash"
        ]
    ]
    .merge(
        plm_duplicate_base[
            [
                "plm_code",
                "Mal grubu",
                "Türkçe malzeme açıklaması",
                "Malzeme Türkçe Adı",
                "Malzeme ingilizce adı",
                "Malzeme statüs"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        [
            "duplicate_group_size",
            "duplicate_group_id",
            "plm_code"
        ],
        ascending=[
            False,
            True,
            True
        ]
    )
)

display(
    duplicate_review.head(100)
)

duplicate_impact_by_group = (
    plm_duplicate_base[
        [
            "plm_code",
            "Mal grubu"
        ]
    ]
    .merge(
        exact_duplicate_plms[
            [
                "plm_code",
                "duplicate_group_id"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .groupby("Mal grubu")
    .agg(
        plm_codes=(
            "plm_code",
            "nunique"
        ),
        duplicate_plm_codes=(
            "duplicate_group_id",
            lambda x: x.notna().sum()
        ),
        duplicate_groups=(
            "duplicate_group_id",
            "nunique"
        )
    )
)

duplicate_impact_by_group[
    "duplicate_plm_rate"
] = (
    duplicate_impact_by_group[
        "duplicate_plm_codes"
    ]
    / duplicate_impact_by_group[
        "plm_codes"
    ]
)

display(
    duplicate_impact_by_group
    .sort_values(
        "duplicate_plm_rate",
        ascending=False
    )
)

duplicate_group_summary = (
    exact_duplicate_plms
    .groupby("duplicate_group_id")
    .agg(
        duplicate_group_size=(
            "plm_code",
            "nunique"
        )
    )
    .reset_index()
)

display(
    duplicate_group_summary[
        "duplicate_group_size"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


In [ ]:
# Normalize PLM status for analysis
plm_status_audit = (
    plm_duplicate_base[
        [
            "plm_code",
            "Mal grubu",
            "Malzeme statüs"
        ]
    ]
    .copy()
)

plm_status_audit["status"] = (
    plm_status_audit["Malzeme statüs"]
    .astype("string")
    .str.strip()
    .str.upper()
    .fillna("<MISSING>")
)

status_distribution = (
    plm_status_audit["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="plm_codes")
)

status_distribution["rate"] = (
    status_distribution["plm_codes"]
    / len(plm_status_audit)
)

display(status_distribution)

# Add status to PLMs that belong to repeated exact technical profiles
# Same data, clearer terminology:
# these are repeated exact observed profiles, not yet confirmed duplicates
exact_profile_plms = exact_duplicate_plms.copy()
profile_status_audit = (
    exact_profile_plms[
        [
            "plm_code",
            "duplicate_group_id"
        ]
    ]
    .merge(
        plm_status_audit[
            [
                "plm_code",
                "Mal grubu",
                "status"
            ]
        ],
        on="plm_code",
        how="left",
        validate="many_to_one"
    )
)

profile_status_summary = (
    profile_status_audit["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="plm_codes")
)

profile_status_summary["rate"] = (
    profile_status_summary["plm_codes"]
    / len(profile_status_audit)
)

display(profile_status_summary)

print(
    "PLMs inside repeated profiles:",
    len(profile_status_audit)
)

print(
    "INCONCEPT PLMs inside repeated profiles:",
    profile_status_audit["status"]
    .eq("INCONCEPT")
    .sum()
)

print(
    "INCONCEPT rate inside repeated profiles:",
    f"{profile_status_audit['status'].eq('INCONCEPT').mean():.2%}"
)


In [ ]:
profile_status_by_group = (
    profile_status_audit
    .groupby("duplicate_group_id")
    .agg(
        profile_size=("plm_code", "nunique"),

        inconcept_count=(
            "status",
            lambda x: (x == "INCONCEPT").sum()
        ),

        distinct_statuses=(
            "status",
            lambda x: tuple(sorted(set(x)))
        )
    )
    .reset_index()
)

profile_status_by_group["inconcept_rate"] = (
    profile_status_by_group["inconcept_count"]
    / profile_status_by_group["profile_size"]
)

profile_status_by_group["profile_status_type"] = np.select(
    [
        profile_status_by_group["inconcept_rate"].eq(1),
        profile_status_by_group["inconcept_rate"].eq(0)
    ],
    [
        "ALL_INCONCEPT",
        "NO_INCONCEPT"
    ],
    default="MIXED"
)

display(
    profile_status_by_group[
        "profile_status_type"
    ]
    .value_counts()
    .rename_axis("profile_status_type")
    .reset_index(name="profile_groups")
)

# Build PLM completeness table cleanly
plm_completeness = (
    plm_duplicate_master[
        ["plm_code"]
        + structured_duplicate_features
        + mv_columns
    ]
    .copy()
)

plm_completeness["structured_filled_count"] = (
    plm_completeness[
        structured_duplicate_features
    ]
    .apply(
        lambda row: sum(
            value != "<MISSING>"
            for value in row
        ),
        axis=1
    )
)

plm_completeness["mv_filled_count"] = (
    plm_completeness[
        mv_columns
    ]
    .apply(
        lambda row: sum(
            value not in {
                "<ABSENT>",
                "<MISSING>"
            }
            for value in row
        ),
        axis=1
    )
)

plm_completeness["technical_filled_count"] = (
    plm_completeness["structured_filled_count"]
    + plm_completeness["mv_filled_count"]
)

plm_completeness["technical_completeness_rate"] = (
    plm_completeness["technical_filled_count"]
    / (
        len(structured_duplicate_features)
        + len(mv_columns)
    )
)

plm_completeness = (
    plm_completeness
    .merge(
        plm_status_audit[
            [
                "plm_code",
                "status"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

assert "Mal grubu" in plm_completeness.columns
assert "status" in plm_completeness.columns
assert plm_completeness["plm_code"].is_unique

plm_completeness["status_bucket"] = np.select(
    [
        plm_completeness["status"] == "INCONCEPT",
        plm_completeness["status"] == "<MISSING>"
    ],
    [
        "INCONCEPT",
        "MISSING"
    ],
    default="OTHER_KNOWN"
)


In [ ]:
fabric_completeness = (
    plm_completeness[
        plm_completeness["Mal grubu"]
        .isin(
            [
                "ORME",
                "DOKUMA",
                "DENIM"
            ]
        )
    ]
    .copy()
)

fabric_status_completeness = (
    fabric_completeness
    .groupby(
        [
            "Mal grubu",
            "status_bucket"
        ]
    )
    .agg(
        plm_codes=(
            "plm_code",
            "nunique"
        ),
        mean_structured_fields=(
            "structured_filled_count",
            "mean"
        ),
        mean_mv_fields=(
            "mv_filled_count",
            "mean"
        ),
        mean_total_fields=(
            "technical_filled_count",
            "mean"
        ),
        median_total_fields=(
            "technical_filled_count",
            "median"
        ),
        mean_completeness_rate=(
            "technical_completeness_rate",
            "mean"
        )
    )
    .reset_index()
)

display(fabric_status_completeness)

display(
    fabric_completeness
    .groupby(
        [
            "Mal grubu",
            "status_bucket"
        ]
    )[
        "technical_filled_count"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

sap_used_plm = (
    sap_plm_mapping[
        ["PLM Kodu"]
    ]
    .dropna()
    .copy()
)

sap_used_plm["plm_code"] = (
    normalize_plm_code_identity(
        sap_used_plm["PLM Kodu"]
    )
)

sap_used_unique = (
    sap_used_plm[
        ["plm_code"]
    ]
    .drop_duplicates()
    .merge(
        plm_status_audit[
            [
                "plm_code",
                "Mal grubu",
                "status"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Unique SAP-used PLM codes:",
    len(sap_used_unique)
)

print(
    "Found in PLM master:",
    sap_used_unique["Mal grubu"]
    .notna()
    .sum()
)

print(
    "Not found in PLM master:",
    sap_used_unique["Mal grubu"]
    .isna()
    .sum()
)


In [ ]:
# Attach technical completeness to repeated exact-profile PLMs
exact_profile_quality = (
    exact_profile_plms[
        [
            "plm_code",
            "duplicate_group_id",
            "duplicate_group_size"
        ]
    ]
    .merge(
        plm_completeness[
            [
                "plm_code",
                "Mal grubu",
                "structured_filled_count",
                "mv_filled_count",
                "technical_filled_count",
                "technical_completeness_rate"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
)

# All PLMs in the same exact profile must have the same technical completeness
profile_quality_summary = (
    exact_profile_quality
    .groupby(
        [
            "duplicate_group_id",
            "Mal grubu"
        ],
        as_index=False
    )
    .agg(
        profile_size=(
            "plm_code",
            "nunique"
        ),
        structured_filled_count=(
            "structured_filled_count",
            "first"
        ),
        mv_filled_count=(
            "mv_filled_count",
            "first"
        ),
        technical_filled_count=(
            "technical_filled_count",
            "first"
        ),
        technical_completeness_rate=(
            "technical_completeness_rate",
            "first"
        )
    )
)

display(
    profile_quality_summary.head()
)

fabric_profile_quality = (
    profile_quality_summary[
        profile_quality_summary["Mal grubu"]
        .isin(
            [
                "ORME",
                "DOKUMA",
                "DENIM"
            ]
        )
    ]
    .copy()
)

display(
    fabric_profile_quality
    .groupby("Mal grubu")[
        [
            "profile_size",
            "structured_filled_count",
            "mv_filled_count",
            "technical_filled_count"
        ]
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)


In [ ]:
profile_information_flags = (
    fabric_profile_quality.copy()
)

profile_information_flags["no_mv_information"] = (
    profile_information_flags[
        "mv_filled_count"
    ] == 0
)

profile_information_flags["only_few_structured_fields"] = (
    profile_information_flags[
        "structured_filled_count"
    ] <= 3
)

profile_information_flags["very_low_information"] = (
    (
        profile_information_flags[
            "mv_filled_count"
        ] == 0
    )
    &
    (
        profile_information_flags[
            "structured_filled_count"
        ] <= 3
    )
)

display(
    profile_information_flags
    .groupby("Mal grubu")
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        no_mv_groups=(
            "no_mv_information",
            "sum"
        ),
        very_low_information_groups=(
            "very_low_information",
            "sum"
        )
    )
)

fabric_all_plm_completeness = (
    plm_completeness[
        plm_completeness["Mal grubu"]
        .isin(
            [
                "ORME",
                "DOKUMA",
                "DENIM"
            ]
        )
    ]
    .copy()
)

group_completeness_thresholds = (
    fabric_all_plm_completeness
    .groupby("Mal grubu")[
        "technical_filled_count"
    ]
    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75
        ]
    )
    .unstack()
    .rename(
        columns={
            0.10: "q10",
            0.25: "q25",
            0.50: "q50",
            0.75: "q75"
        }
    )
    .reset_index()
)

display(group_completeness_thresholds)

profile_information_flags = (
    profile_information_flags
    .merge(
        group_completeness_thresholds,
        on="Mal grubu",
        how="left",
        validate="many_to_one"
    )
)

profile_information_flags["information_level"] = np.select(
    [
        profile_information_flags[
            "technical_filled_count"
        ] <= profile_information_flags["q10"],

        profile_information_flags[
            "technical_filled_count"
        ] <= profile_information_flags["q25"],

        profile_information_flags[
            "technical_filled_count"
        ] >= profile_information_flags["q75"]
    ],
    [
        "VERY_LOW_INFORMATION",
        "LOW_INFORMATION",
        "HIGH_INFORMATION"
    ],
    default="NORMAL_INFORMATION"
)


In [ ]:
profile_information_summary = (
    profile_information_flags
    .groupby(
        [
            "Mal grubu",
            "information_level"
        ]
    )
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        plm_codes=(
            "profile_size",
            "sum"
        ),
        median_profile_size=(
            "profile_size",
            "median"
        ),
        max_profile_size=(
            "profile_size",
            "max"
        ),
        median_filled_fields=(
            "technical_filled_count",
            "median"
        )
    )
    .reset_index()
)

display(profile_information_summary)

display(
    profile_information_flags
    .sort_values(
        [
            "profile_size",
            "technical_filled_count"
        ],
        ascending=[
            False,
            False
        ]
    )
    [
        [
            "duplicate_group_id",
            "Mal grubu",
            "profile_size",
            "structured_filled_count",
            "mv_filled_count",
            "technical_filled_count",
            "information_level"
        ]
    ]
    .head(40)
)

high_information_knit_groups = (
    profile_information_flags[
        (
            profile_information_flags[
                "Mal grubu"
            ] == "ORME"
        )
        &
        (
            profile_information_flags[
                "information_level"
            ] == "HIGH_INFORMATION"
        )
    ]
    .sort_values(
        "profile_size",
        ascending=False
    )
)

display(
    high_information_knit_groups.head(10)
)

profile_information_flags["duplicate_evidence_class"] = (
    profile_information_flags["information_level"]
    .map({
        "VERY_LOW_INFORMATION": "INSUFFICIENT_PROFILE",
        "LOW_INFORMATION": "INSUFFICIENT_PROFILE",
        "NORMAL_INFORMATION": "STRONG_PROFILE_MATCH",
        "HIGH_INFORMATION": "VERY_STRONG_PROFILE_MATCH"
    })
)

display(
    profile_information_flags
    .groupby(
        [
            "Mal grubu",
            "duplicate_evidence_class"
        ]
    )
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        plm_codes=(
            "profile_size",
            "sum"
        )
    )
)


In [ ]:
def normalize_plm_join_key(series):
    result = (
        series
        .astype("string")
        .str.normalize("NFKC")
        .str.strip()
    )

    numeric_decimal_mask = (
        result.str.fullmatch(
            r"\d+\.0+",
            na=False
        )
    )

    result.loc[numeric_decimal_mask] = (
        result.loc[numeric_decimal_mask]
        .str.replace(
            r"\.0+$",
            "",
            regex=True
        )
    )

    return result

profile_plm_members = (
    exact_profile_plms[
        [
            "plm_code",
            "duplicate_group_id"
        ]
    ]
    .copy()
)

profile_plm_members["plm_join_key"] = (
    normalize_plm_join_key(
        profile_plm_members["plm_code"]
    )
)

profile_plm_members = (
    profile_plm_members
    .merge(
        profile_information_flags[
            [
                "duplicate_group_id",
                "Mal grubu",
                "profile_size",
                "technical_filled_count",
                "information_level",
                "duplicate_evidence_class"
            ]
        ],
        on="duplicate_group_id",
        how="left",
        validate="many_to_one"
    )
)

sap_plm_usage = (
    sap_plm_mapping[
        [
            "Malzeme",
            "PLM Kodu"
        ]
    ]
    .dropna(subset=["PLM Kodu"])
    .copy()
)

sap_plm_usage["plm_join_key"] = (
    normalize_plm_join_key(
        sap_plm_usage["PLM Kodu"]
    )
)

sap_usage_by_plm = (
    sap_plm_usage
    .groupby("plm_join_key")
    .agg(
        sap_material_count=(
            "Malzeme",
            "nunique"
        )
    )
    .reset_index()
)

profile_plm_usage = (
    profile_plm_members
    .merge(
        sap_usage_by_plm,
        on="plm_join_key",
        how="left",
        validate="many_to_one"
    )
)

profile_plm_usage["sap_material_count"] = (
    profile_plm_usage[
        "sap_material_count"
    ]
    .fillna(0)
    .astype(int)
)


In [ ]:
profile_usage_summary = (
    profile_plm_usage
    .groupby(
        [
            "duplicate_group_id",
            "Mal grubu",
            "duplicate_evidence_class"
        ],
        as_index=False
    )
    .agg(
        profile_size=(
            "plm_code",
            "nunique"
        ),

        used_plm_codes=(
            "sap_material_count",
            lambda x: (x > 0).sum()
        ),

        total_sap_materials=(
            "sap_material_count",
            "sum"
        ),

        max_sap_materials_on_one_plm=(
            "sap_material_count",
            "max"
        )
    )
)

profile_usage_summary["unused_plm_codes"] = (
    profile_usage_summary["profile_size"]
    - profile_usage_summary["used_plm_codes"]
)

profile_usage_summary["usage_pattern"] = np.select(
    [
        profile_usage_summary["used_plm_codes"] == 0,
        profile_usage_summary["used_plm_codes"] == 1,
        profile_usage_summary["used_plm_codes"] > 1
    ],
    [
        "NONE_USED_IN_SAP",
        "ONE_PLM_USED_IN_SAP",
        "MULTIPLE_PLM_USED_IN_SAP"
    ],
    default="UNKNOWN"
)

display(
    profile_usage_summary[
        profile_usage_summary[
            "Mal grubu"
        ].isin(
            ["ORME", "DOKUMA", "DENIM"]
        )
    ]
    .groupby(
        [
            "Mal grubu",
            "duplicate_evidence_class",
            "usage_pattern"
        ]
    )
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        total_plm_codes=(
            "profile_size",
            "sum"
        ),
        total_sap_materials=(
            "total_sap_materials",
            "sum"
        )
    )
    .reset_index()
)

priority_duplicate_groups = (
    profile_usage_summary[
        (
            profile_usage_summary[
                "duplicate_evidence_class"
            ].isin([
                "STRONG_PROFILE_MATCH",
                "VERY_STRONG_PROFILE_MATCH"
            ])
        )
        &
        (
            profile_usage_summary[
                "usage_pattern"
            ] == "MULTIPLE_PLM_USED_IN_SAP"
        )
    ]
    .copy()
)

display(
    priority_duplicate_groups
    .sort_values(
        [
            "Mal grubu",
            "total_sap_materials"
        ],
        ascending=[True, False]
    )
)


In [ ]:
priority_duplicate_plms = (
    profile_plm_usage[
        profile_plm_usage[
            "duplicate_group_id"
        ].isin(
            priority_duplicate_groups[
                "duplicate_group_id"
            ]
        )
    ]
    .merge(
        plm_duplicate_base[
            [
                "plm_code",
                "Türkçe malzeme açıklaması",
                "Malzeme Türkçe Adı",
                "Malzeme ingilizce adı"
            ]
        ],
        on="plm_code",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        [
            "duplicate_group_id",
            "sap_material_count",
            "plm_code"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
)

display(
    priority_duplicate_plms[
        [
            "duplicate_group_id",
            "Mal grubu",
            "duplicate_evidence_class",
            "profile_size",
            "plm_code",
            "sap_material_count",
            "Türkçe malzeme açıklaması",
            "Malzeme Türkçe Adı",
            "Malzeme ingilizce adı"
        ]
    ]
)

priority_duplicate_sap_materials = (
    sap_plm_usage
    .merge(
        priority_duplicate_plms[
            [
                "duplicate_group_id",
                "Mal grubu",
                "duplicate_evidence_class",
                "plm_join_key",
                "plm_code"
            ]
        ],
        on="plm_join_key",
        how="inner",
        validate="many_to_one"
    )
)

priority_duplicate_sap_materials = (
    priority_duplicate_sap_materials
    .merge(
        sap_plm_mapping[
            [
                "Malzeme",
                "Türkçe malzeme açıklaması",
                "Türkçe malzeme Uzun açıklaması"
            ]
        ]
        .drop_duplicates("Malzeme"),
        on="Malzeme",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        [
            "duplicate_group_id",
            "plm_code",
            "Malzeme"
        ]
    )
)

display(
    priority_duplicate_sap_materials[
        [
            "duplicate_group_id",
            "Mal grubu",
            "plm_code",
            "Malzeme",
            "Türkçe malzeme açıklaması",
            "Türkçe malzeme Uzun açıklaması"
        ]
    ]
)


In [ ]:
single_used_cleanup_groups = (
    profile_usage_summary[
        (
            profile_usage_summary[
                "duplicate_evidence_class"
            ].isin([
                "STRONG_PROFILE_MATCH",
                "VERY_STRONG_PROFILE_MATCH"
            ])
        )
        &
        (
            profile_usage_summary[
                "usage_pattern"
            ] == "ONE_PLM_USED_IN_SAP"
        )
    ]
    .copy()
)

display(
    single_used_cleanup_groups
    .groupby("Mal grubu")
    .agg(
        profile_groups=(
            "duplicate_group_id",
            "nunique"
        ),
        total_plm_codes=(
            "profile_size",
            "sum"
        ),
        total_sap_materials=(
            "total_sap_materials",
            "sum"
        )
    )
)


## 9. Limitations and next steps

- **Private data dependency:** the repository does not include source Excel files, so public users cannot reproduce the results without equivalent schemas.
- **Source-schema coupling:** several Turkish column names are intentionally preserved because they are exact fields in the upstream exports.
- **Domain rules:** scope keywords, material-group mappings, and confidence thresholds are business-context assumptions and should be reviewed when the source systems change.
- **No-attribute records:** the fallback pipeline is materially weaker than the structured-attribute path, especially where descriptions are ambiguous.
- **Duplicate detection:** repeated exact technical profiles are review candidates, not proof that records should be merged.

Recommended next steps are to add a small synthetic public dataset for reproducibility, convert reusable transformations into a `src/` package, and add unit tests for identifier normalization, yarn parsing, weight scaling, and candidate uniqueness.
